# Mapping the Cuisine Landscape: How Are Restaurants Truly Related?

**How does the *way* you represent food language determine which cuisines appear close to each other?**

### The Core Objective
In this task, we move beyond individual reviews to ask a structural question: are Italian and Mediterranean restaurants semantically similar? Is "Fast Food" its own world, or does it overlap with American cuisine? To answer this, we aggregate all reviews for each cuisine or subcategory into a single "document" and measure similarity with two complementary representations, then use clustering to test whether those similarities form coherent category neighborhoods. The representation choice turns out to matter enormously.

### Our Two Representations plus Clustering
1. **The Discrimination Representation (TF-IDF):** Weighted term frequency ? which words are *specific* to a cuisine, discounting terms that appear everywhere?
2. **The Geometry Representation (Sentence Embeddings):** Dense vector representations ? which cuisines occupy nearby regions of semantic space, even when they use different vocabulary?
3. **The Clustering Check:** Density-aware clustering on the embedding space ? do the category neighborhoods look stable enough to support later dish and restaurant analysis?

### Why This Matters
The cuisine similarity map built here serves as the structural backbone for later tasks. **Task 3 (Dish/Phrase Discovery)** uses your category clusters to focus phrase mining on a coherent target; **Task 4 (Restaurant Ranking)** needs cuisine neighborhood context to interpret dish mentions and sentiment; **Task 5 (Temporal Analysis)** can track whether cuisine relationships shift over time. Getting the representation right in Task 2 pays dividends throughout the rest of the capstone.

## Table of Contents

1. [Runtime and Package Setup](#0.-Runtime-and-package-setup)
2. [Configuration](#1.-Configuration)
3. [Dataset Acquisition and Discovery](#2.-Dataset-acquisition-and-discovery)
4. [Build the Cuisine/Subcategory Subset](#3.-Build-the-cuisine/subcategory-review-subset)
5. [Pipeline A — Classical Bag-of-Words Similarity](#4.-Pipeline-A---Classical-bag-of-words-similarity)
6. [Pipeline B — Modern Embedding-Based Similarity](#5.-Pipeline-B---Modern-embedding-based-similarity)
7. [Clustering](#6.-Clustering)
8. [Downstream Handoff Artifacts](#Downstream-handoff:-cuisine/subcategory-map-artifacts)
9. [Visual Evidence and Interpretation](#Visual-Evidence-and-Interpretation)
10. [Required Comparison and Interpretation](#Required-comparison-and-interpretation)
11. [Required Report Deliverables](#8.-Required-report-deliverables)


# Task 2 - Cuisine/Subcategory Similarity & Clustering

Task 1 looked at themes in individual reviews and rating-based subsets. Task 2 is the map-building component of the Yelp restaurant intelligence system. It moves one level up: instead of treating each review as the unit of analysis, you will aggregate reviews by **cuisine or food/restaurant subcategory** and ask:

- How similar are cuisines/subcategories based on what people say in reviews?
- How much does the similarity map change when you use classical bag-of-words representations versus modern embeddings?
- What clusters of cuisines/subcategories emerge, and are they interpretable?

The core intellectual goal is not merely to run clustering. The goal is to understand how **representation choices** affect the structure you discover.

These maps help later tasks choose coherent cuisines/categories for phrase discovery, dish search, temporal monitoring, and prediction reflection.

## Pipeline Artifact Handoff

Task 2 produces cuisine/subcategory similarity and cluster artifacts for the capstone pipeline. Its key handoff artifacts live under `artifacts/task2/`, especially `subcategory_similarity_tfidf.csv`, `subcategory_similarity_embedding.csv`, `subcategory_clusters.csv`, and `cuisine_map_summary.csv`. Later tasks can use these artifacts to choose coherent categories and interpret restaurant/cuisine structure.

`smoke_test` mode is for smoke-test validation only. Student reports should use the real assignment data path. TF-IDF is the required classical baseline; embedding representations are the modern comparison path, and defensive offline fallbacks are not substitutes for required modern analysis in real submissions.


## Required artifact contract for Task 2

Your completed notebook should produce these core artifacts under `artifacts/task2/`:

- `subcategory_similarity_tfidf.csv` — TF-IDF cosine similarity matrix.
- `subcategory_similarity_embedding.csv` — embedding cosine similarity matrix (or labeled fallback).
- `subcategory_clusters.csv` — cluster assignments from HDBSCAN.
- `cuisine_map_summary.csv` — final comparison summary for your report.

Optional figure artifacts are welcome, but the above are the minimum peer-gradable outputs.

## Learning objectives

By completing this task, you should be able to:

1. Build category-level documents by aggregating restaurant reviews.
2. Compute pairwise similarity with a classical TF-IDF representation.
3. Compute pairwise similarity with a modern embedding-based representation.
4. Visualize similarity matrices and a 2-D category map.
5. Apply one clustering method to the selected representation.
6. Explain how representation choice affects interpretability.

## 0. Runtime and package setup

### Local installation and run

You can download and run this notebook on your personal computer, which will likely have a faster CPU than what you would get on Google Colab. We highly recommend running locally for the embedding-heavy sections of this task.

[Jupyter Installation Instructions](https://docs.jupyter.org/en/latest/install.html)

### Google Colab

You can also run this notebook on Google Colab, but be aware that Colab turns off the underlying cloud resources after a set amount of time, at which point you will need to rerun all cells (including `pip install` and dataset downloads).

- Use **Runtime → Change runtime type → Python 3**.
- A GPU is optional but speeds up sentence-transformer embedding significantly.
- Start with modest sample sizes; you can increase them once the full pipeline runs.

The modern embedding pipeline uses `sentence-transformers` in real assignment runs. Defensive fallback code is retained so smoke-test validation can run without network access or model downloads, but fallback embeddings are not a substitute for the required modern sentence-transformer comparison in real student work.


## *Smoke Test Mode*

*Use `smoke_test` mode only to verify that your notebook runs end-to-end when real data is unavailable. Base your final conclusions on the real Yelp dataset and clearly label any fallback method outputs. Fallback embeddings produced in smoke mode are not a substitute for the required sentence-transformer comparison in real submissions.*


In [ ]:
# Optional package installation for Colab.
# Set INSTALL_OPTIONAL_PACKAGES = False when running in a fresh Colab runtime.
INSTALL_OPTIONAL_PACKAGES = True

if INSTALL_OPTIONAL_PACKAGES:
    import sys, subprocess
    pkgs = [
        "kagglehub",
        "pyarrow",
        "sentence-transformers",
        "umap-learn",
        "hdbscan",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

In [ ]:
import os
import re
import json
import tarfile
import zipfile
import math
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import normalize, StandardScaler

RANDOM_SEED = 3847
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

### Xiaofei
### 0.1 Local path bootstrap (run before Configuration)

This cell makes Task 2 runnable from any machine or Colab session:

1. **Resolve the project root** — walk up from the current working directory (or use `DMC_PROJECT_ROOT`) until a folder containing `notebooks/` or `artifacts/` is found, then `chdir` there so relative paths like `data/raw` and `artifacts/task2` resolve correctly.
2. **Set default data paths for real Yelp runs** — `TASK2_DATA_SOURCE=manual` and `TASK2_DATA_DIR=data/raw` (the same local copy used in Task 1).

Run this cell **once per kernel**, immediately after the import cell and **before** Section 1 Configuration. To override defaults, set environment variables before running: `TASK2_DATA_SOURCE`, `TASK2_DATA_DIR`, or `DMC_PROJECT_ROOT`.

In [ ]:
# Xiaofei's code
# this cell is to ensure the notebook runs on any machine / Colab

import os
from pathlib import Path

# create a function to help to find the project root
def find_project_root(start: Path) -> Path:
    """Walk up from `start` until we find the repo root (contains artifacts/ or notebooks/)."""
    for candidate in [start, *start.parents]:
        if (candidate / "artifacts").is_dir() or (candidate / "notebooks").is_dir():
            return candidate
    return start


# Resolve the project root portably so the notebook runs on any machine / Colab:
# explicit override via the DMC_PROJECT_ROOT environment variable, else
# auto-detect by walking up from the current working directory.
env_root=os.environ.get("DMC_PROJECT_ROOT", "").strip()
if env_root:
    PROJECT_ROOT=Path(env_root).expanduser().resolve()
else:
    PROJECT_ROOT=find_project_root(Path.cwd().resolve())
# change the working directory to the project root
os.chdir(PROJECT_ROOT)
# set the environment variables
os.environ.setdefault("TASK2_DATA_SOURCE", "manual")
os.environ.setdefault("TASK2_DATA_DIR", "data/raw")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", os.getcwd())
print("DATA_SOURCE will be:", os.environ.get("TASK2_DATA_SOURCE"))
print("TASK2_DATA_DIR will be:", os.environ.get("TASK2_DATA_DIR"))

## 1. Configuration

The modern Yelp dataset is large, so the streaming and sampling controls in this notebook matter. The default category focus is `cuisine_focused`, which keeps the analysis centered on restaurant/cuisine/food subcategories rather than mixing cuisines with beverage retail, venues, and generic Yelp parent labels. You may set `TASK2_CATEGORY_FOCUS_MODE=broad` for an extension, but your report should explain why the broader taxonomy is still coherent.

The review sampler first builds a bounded candidate pool and then caps the number of reviews contributed by any one business. This makes the cuisine/subcategory map less dependent on a few heavily reviewed restaurants while still preserving enough review text for stable similarity estimates.

In real data modes, sentence-transformer embeddings are the required modern comparison path. TF-IDF is the required classical baseline. Fallback representations are only for smoke-test or constrained environments, and must be labeled clearly in the report.

In [ ]:
# Dataset source options:
#   "kaggle_latest" -> kagglehub.dataset_download("yelp-dataset/yelp-dataset")
#   "manual"        -> manually set DATA_DIR to a directory containing Yelp JSON files
#   "smoke_test"    -> synthetic Yelp-shaped data for validation without internet
DATA_SOURCE = os.environ.get("TASK2_DATA_SOURCE", "smoke_test")
DATA_DIR = os.environ.get("TASK2_DATA_DIR", "")
SMOKE_DATA_DIR = Path(os.environ.get("TASK2_SMOKE_DATA_DIR", Path("data") / "smoke_task2"))

# Gate businesses using broad restaurant/food markers, then keep more specific subcategories.
GATE_TOKENS = {"Restaurants", "Food"}
REQUIRE_RESTAURANTS = True  # keep the capstone focused on restaurant/food review mining
CATEGORY_FOCUS_MODE = os.environ.get("TASK2_CATEGORY_FOCUS_MODE", "cuisine_focused").strip().lower()

# Review collection controls. Increase MAX_REVIEWS for real course runs if Colab memory allows.
MAX_REVIEWS = int(os.environ.get("TASK2_MAX_REVIEWS", "80000"))
# Build a larger candidate pool, then cap each business before the final sample.
REVIEW_CANDIDATE_MULTIPLIER = int(os.environ.get("TASK2_REVIEW_CANDIDATE_MULTIPLIER", "3"))
MAX_REVIEW_CANDIDATES = int(os.environ.get("TASK2_MAX_REVIEW_CANDIDATES", str(MAX_REVIEWS * REVIEW_CANDIDATE_MULTIPLIER)))
MAX_CANDIDATE_REVIEWS_PER_BUSINESS = int(os.environ.get("TASK2_MAX_CANDIDATE_REVIEWS_PER_BUSINESS", "120"))
MAX_FINAL_REVIEWS_PER_BUSINESS = int(os.environ.get("TASK2_MAX_FINAL_REVIEWS_PER_BUSINESS", "60"))
MIN_REVIEWS_PER_SUBCATEGORY = int(os.environ.get("TASK2_MIN_REVIEWS_PER_SUBCATEGORY", "250"))
TOP_N_SUBCATEGORIES = int(os.environ.get("TASK2_TOP_N_SUBCATEGORIES", "30"))
MAX_TEXT_CHARS_PER_SUBCATEGORY = int(os.environ.get("TASK2_MAX_TEXT_CHARS_PER_SUBCATEGORY", "250000"))

# Vectorization/modeling controls
MAX_FEATURES = 6000
NGRAM_RANGE = (1, 2)
N_CLUSTERS = 6
MAX_SNIPPETS_PER_SUBCATEGORY = 200
SNIPPET_CHAR_LIMIT = 600
SENTENCE_TRANSFORMER_MODEL = os.environ.get("TASK2_SENTENCE_TRANSFORMER_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
MODEL_CACHE_DIR = os.environ.get("TASK2_MODEL_CACHE_DIR", "")
LOCAL_FILES_ONLY = os.environ.get("TASK2_HF_LOCAL_FILES_ONLY", "0").strip().lower() in {"1", "true", "yes"}
REQUIRE_SENTENCE_TRANSFORMER = os.environ.get(
    "TASK2_REQUIRE_SENTENCE_TRANSFORMER",
    "0" if DATA_SOURCE.strip().lower() == "smoke_test" else "1",
).strip().lower() in {"1", "true", "yes"}
ENABLE_PRIMARY_CATEGORY_SENSITIVITY = os.environ.get("TASK2_ENABLE_PRIMARY_CATEGORY_SENSITIVITY", "1").strip().lower() in {"1", "true", "yes"}
TOP_TERMS_PER_SUBCATEGORY = int(os.environ.get("TASK2_TOP_TERMS_PER_SUBCATEGORY", "8"))

OUTPUT_DIR = Path(os.environ.get("TASK2_ARTIFACT_DIR", Path("artifacts") / "task2"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if MODEL_CACHE_DIR:
    Path(MODEL_CACHE_DIR).mkdir(parents=True, exist_ok=True)

print("DATA_SOURCE:", DATA_SOURCE)
print("MAX_REVIEWS:", MAX_REVIEWS)
print("TOP_N_SUBCATEGORIES:", TOP_N_SUBCATEGORIES)
print("MAX_REVIEW_CANDIDATES:", MAX_REVIEW_CANDIDATES)
print("MAX_CANDIDATE_REVIEWS_PER_BUSINESS:", MAX_CANDIDATE_REVIEWS_PER_BUSINESS)
print("MAX_FINAL_REVIEWS_PER_BUSINESS:", MAX_FINAL_REVIEWS_PER_BUSINESS)
print("CATEGORY_FOCUS_MODE:", CATEGORY_FOCUS_MODE)
print("SENTENCE_TRANSFORMER_MODEL:", SENTENCE_TRANSFORMER_MODEL)
print("TASK2_HF_LOCAL_FILES_ONLY:", LOCAL_FILES_ONLY)
print("TASK2_REQUIRE_SENTENCE_TRANSFORMER:", REQUIRE_SENTENCE_TRANSFORMER)
print("TASK2_ENABLE_PRIMARY_CATEGORY_SENSITIVITY:", ENABLE_PRIMARY_CATEGORY_SENSITIVITY)

## 2. Dataset acquisition and discovery

The loader searches for Yelp-style business and review JSON files. It also handles the common Kaggle case where the downloaded directory contains a nested `.zip` archive rather than extracted JSON files.

In [ ]:
def extract_archives_if_needed(dataset_dir: Path) -> Path:
    """Extract common archives if JSON files are not already visible."""
    dataset_dir = Path(dataset_dir)
    json_files = list(dataset_dir.rglob("*.json"))
    if json_files:
        return dataset_dir

    archives = list(dataset_dir.rglob("*.zip")) + list(dataset_dir.rglob("*.tar")) + list(dataset_dir.rglob("*.tar.gz")) + list(dataset_dir.rglob("*.tgz"))
    if not archives:
        return dataset_dir

    extract_dir = dataset_dir / "_extracted"
    extract_dir.mkdir(exist_ok=True)
    for archive in archives:
        print(f"Extracting nested archive: {archive.name}")
        try:
            if archive.suffix == ".zip":
                with zipfile.ZipFile(archive, "r") as zf:
                    zf.extractall(extract_dir)
            else:
                with tarfile.open(archive, "r:*") as tf:
                    tf.extractall(extract_dir)
        except Exception as e:
            print(f"Warning: could not extract {archive}: {e}")
    return dataset_dir


def download_or_locate_dataset(data_source: str, data_dir: str = "") -> Path:
    data_source = data_source.lower().strip()
    if data_source == "manual":
        if not data_dir:
            raise ValueError("For DATA_SOURCE='manual', set DATA_DIR to the dataset directory.")
        return extract_archives_if_needed(Path(data_dir))

    if data_source == "kaggle_latest":
        try:
            import kagglehub
        except ImportError as e:
            raise ImportError("Install kagglehub or set DATA_SOURCE='manual'/'smoke_test'.") from e
        path = Path(kagglehub.dataset_download("yelp-dataset/yelp-dataset"))
        return extract_archives_if_needed(path)

    if data_source == "smoke_test":
        return create_smoke_test_dataset(SMOKE_DATA_DIR)

    raise ValueError(f"Unknown DATA_SOURCE: {data_source}")


def find_yelp_json_file(dataset_dir: Path, kind: str) -> Path:
    """Find a business or review JSON file under dataset_dir."""
    dataset_dir = Path(dataset_dir)
    candidates = list(dataset_dir.rglob("*.json"))
    if kind == "business":
        patterns = ["business"]
    elif kind == "review":
        patterns = ["review"]
    else:
        raise ValueError(kind)

    matches = [p for p in candidates if all(token in p.name.lower() for token in patterns)]
    if not matches:
        raise FileNotFoundError(f"Could not find a Yelp {kind} JSON file under {dataset_dir}. Found: {[p.name for p in candidates[:10]]}")
    # Prefer academic dataset names if available.
    matches = sorted(matches, key=lambda p: ("academic" not in p.name.lower(), len(str(p))))
    return matches[0]

### 2.1 Smoke-test fixture used only for validation

This creates a small Yelp-shaped dataset with category-specific vocabulary. It is intentionally simple and separable; it proves the notebook executes, but it is not a substitute for the real Yelp dataset.

In [ ]:
def create_smoke_test_dataset(out_dir: Path, n_businesses_per_cat: int = 8, reviews_per_business: int = 14) -> Path:
    out_dir.mkdir(exist_ok=True)
    business_path = out_dir / "yelp_academic_dataset_business.json"
    review_path = out_dir / "yelp_academic_dataset_review.json"

    categories = {
        "Mexican": ["tacos", "salsa", "burrito", "tortilla", "guacamole", "carne", "al pastor", "quesadilla"],
        "Chinese": ["noodles", "dumplings", "sichuan", "wonton", "soy sauce", "fried rice", "mapo", "bao"],
        "Italian": ["pasta", "pizza", "risotto", "parmesan", "gnocchi", "marinara", "lasagna", "espresso"],
        "Japanese": ["sushi", "ramen", "miso", "tempura", "sashimi", "udon", "teriyaki", "bento"],
        "Indian": ["curry", "naan", "masala", "tandoori", "biryani", "paneer", "dal", "chutney"],
        "Thai": ["pad thai", "curry", "basil", "lemongrass", "tom yum", "noodles", "coconut", "papaya"],
        "Coffee & Tea": ["latte", "espresso", "cold brew", "pastry", "barista", "tea", "croissant", "wifi"],
        "Bakeries": ["bread", "croissant", "cake", "pastry", "cookies", "sourdough", "muffin", "dessert"],
        "Burgers": ["burger", "fries", "cheddar", "patty", "bun", "milkshake", "bacon", "grill"],
        "Seafood": ["oysters", "salmon", "shrimp", "crab", "fish", "clam", "lobster", "ceviche"],
    }
    ambience_terms = ["service", "friendly", "busy", "fresh", "flavor", "portion", "price", "wait", "atmosphere", "recommend"]
    cities = ["Philadelphia", "Tampa", "Tucson", "Nashville", "Reno"]

    businesses = []
    reviews = []
    rid = 0
    for cat, vocab in categories.items():
        for b in range(n_businesses_per_cat):
            bid = f"biz_{cat.replace(' ', '_').replace('&','and')}_{b}"
            extra = "Food" if cat in {"Coffee & Tea", "Bakeries"} else "Restaurants"
            categories_str = f"Restaurants, Food, {cat}"
            businesses.append({
                "business_id": bid,
                "name": f"{cat} Place {b}",
                "city": random.choice(cities),
                "state": "CA",
                "stars": round(random.uniform(3.0, 4.8), 1),
                "review_count": reviews_per_business,
                "categories": categories_str,
            })
            for j in range(reviews_per_business):
                words = random.sample(vocab, k=min(4, len(vocab))) + random.sample(ambience_terms, k=5)
                random.shuffle(words)
                text = f"The {' '.join(words)} made this {cat} visit memorable. I would come back for the {random.choice(vocab)}."
                stars = random.choice([3, 4, 4, 5, 5])
                reviews.append({
                    "review_id": f"rev_{rid}",
                    "user_id": f"user_{rid % 70}",
                    "business_id": bid,
                    "stars": stars,
                    "date": f"2021-{(j % 12) + 1:02d}-{(j % 27) + 1:02d}",
                    "text": text,
                    "useful": random.randint(0, 5),
                    "funny": random.randint(0, 2),
                    "cool": random.randint(0, 3),
                })
                rid += 1

    with business_path.open("w", encoding="utf-8") as f:
        for row in businesses:
            f.write(json.dumps(row) + "\n")
    with review_path.open("w", encoding="utf-8") as f:
        for row in reviews:
            f.write(json.dumps(row) + "\n")

    return out_dir

In [ ]:
dataset_dir = download_or_locate_dataset(DATA_SOURCE, DATA_DIR)
business_path = find_yelp_json_file(dataset_dir, "business")
review_path = find_yelp_json_file(dataset_dir, "review")

print("Dataset directory:", dataset_dir)
print("Business JSON:", business_path)
print("Review JSON:", review_path)

## 3. Build the cuisine/subcategory review subset

We gate on broad `Food`/`Restaurants` categories, then keep more specific subcategories such as `Mexican`, `Italian`, `Thai`, `Coffee & Tea`, `Bakeries`, and so on.

This intentionally preserves the old capstone idea of a **cuisine map**, while allowing a practical modern variant: a **food/restaurant subcategory map**.

Two data-design issues matter here:

1. Yelp review streams can be dominated by a few heavily reviewed businesses. We therefore collect a bounded candidate pool and cap per-business contribution before the final sample.
2. Yelp businesses are multi-label. A single review can belong to both `Italian` and `Pizza`, or both `Japanese` and `Sushi Bars`. We keep the main analysis focused on the selected multi-label category corpus and discuss this as a data-design limitation rather than a separate diagnostic assignment.


### 3.1 Category gate and focused taxonomy

The first modeling choice is not an algorithm; it is the definition of the analysis universe. We gate on restaurant/food businesses and then keep cuisine-like or food-subcategory labels so Task 2 stays a cuisine/category mapping assignment rather than a generic Yelp taxonomy exercise.

For students, this is a place to ask: *Am I comparing coherent categories, or did I accidentally mix cuisines, venues, retail, and parent labels?*

In [ ]:
GENERIC_DROP = {
    "Restaurants", "Food", "Nightlife", "Bars", "Event Planning & Services", "Caterers",
    "American (New)", "American (Traditional)", "Fast Food", "Sandwiches", "Salad",
    "Shopping", "Local Services", "Arts & Entertainment", "Hotels & Travel", "Active Life",
    "Health & Medical", "Beauty & Spas", "Home Services", "Automotive", "Professional Services",
}

# Instructor default: keep the worked solution focused on food/cuisine-like categories.
# The broad Yelp taxonomy can be useful, but it mixes cuisines, venues, beverage retail,
# desserts, and meal occasions, which makes the clustering lesson much noisier.
CUISINE_FOCUS_SUBCATEGORIES = {
    "Seafood", "Pizza", "Burgers", "Mexican", "Italian", "Cajun/Creole",
    "Sushi Bars", "Japanese", "Chinese", "Southern", "Barbeque", "Steakhouses",
    "Vegetarian", "Asian Fusion", "Thai", "Vietnamese", "Korean", "Indian",
    "Mediterranean", "Greek", "Middle Eastern", "French", "Spanish", "Tapas Bars",
    "Caribbean", "Latin American", "Peruvian", "Tex-Mex", "Ramen", "Noodles",
}
MIN_FOCUSED_SUBCATEGORIES = 8

KEEP_REVIEW_COLS = ["review_id", "user_id", "business_id", "stars", "date", "text", "useful", "funny", "cool"]


def normalize_categories(cats) -> list[str]:
    if isinstance(cats, list):
        return [str(c).strip() for c in cats if str(c).strip()]
    if cats is None or (isinstance(cats, float) and pd.isna(cats)):
        return []
    text = str(cats)
    # Yelp academic data usually uses comma-separated category strings.
    parts = re.split(r",|;", text)
    return [p.strip() for p in parts if p.strip()]


def is_eligible_business(cats: list[str]) -> bool:
    cat_set = set(cats)
    if REQUIRE_RESTAURANTS:
        return bool(cat_set & GATE_TOKENS)
    return bool(cat_set)


def plausible_subcategories(cats: list[str]) -> list[str]:
    return [c for c in cats if c not in GENERIC_DROP and len(c) >= 3]


def collect_business_subcategories(business_json_path: Path, chunksize: int = 100_000):
    id_to_subcats = {}
    id_to_meta = {}
    total = eligible = 0

    for chunk in tqdm(pd.read_json(business_json_path, lines=True, chunksize=chunksize), desc="Scanning businesses"):
        total += len(chunk)
        if "business_id" not in chunk.columns or "categories" not in chunk.columns:
            continue
        for row in chunk.itertuples(index=False):
            row_dict = row._asdict() if hasattr(row, "_asdict") else dict(row)
            bid = str(row_dict.get("business_id"))
            cats = normalize_categories(row_dict.get("categories"))
            if not is_eligible_business(cats):
                continue
            subcats = plausible_subcategories(cats)
            if not subcats:
                continue
            eligible += 1
            id_to_subcats[bid] = subcats
            id_to_meta[bid] = {
                "name": row_dict.get("name", ""),
                "city": row_dict.get("city", ""),
                "state": row_dict.get("state", ""),
                "business_stars": row_dict.get("stars", np.nan),
                "business_review_count": row_dict.get("review_count", np.nan),
                "raw_categories": row_dict.get("categories", ""),
            }

    print(f"Scanned {total:,} businesses; retained {eligible:,} food/restaurant businesses with usable subcategories.")
    return id_to_subcats, id_to_meta

### 3.2 Capped streaming sampler

Yelp review streams are not balanced. A few very popular restaurants can contribute hundreds or thousands of reviews, which can make a category vector describe a business brand more than a cuisine. The sampler below first collects a bounded candidate pool and then caps each business before taking the final sample.

This is a data-mining design decision: we want enough text for stable category signals, but we also want broad restaurant coverage.

In [ ]:
def _business_review_count_summary(counts: pd.Series, prefix: str) -> dict:
    if counts.empty:
        return {
            f"{prefix}_businesses": 0,
            f"{prefix}_max_reviews_per_business": 0,
            f"{prefix}_median_reviews_per_business": 0.0,
            f"{prefix}_p95_reviews_per_business": 0.0,
        }
    return {
        f"{prefix}_businesses": int(counts.shape[0]),
        f"{prefix}_max_reviews_per_business": int(counts.max()),
        f"{prefix}_median_reviews_per_business": float(counts.median()),
        f"{prefix}_p95_reviews_per_business": float(counts.quantile(0.95)),
    }


def stream_reviews_for_eligible_businesses(
    review_json_path: Path,
    eligible_biz_ids: set[str],
    biz_to_subcats: dict,
    max_reviews: int,
    max_candidate_reviews: int,
    max_candidate_reviews_per_business: int,
    max_final_reviews_per_business: int,
    chunksize: int = 100_000,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Collect a bounded, capped review sample without letting one business dominate."""
    candidate_cap = max(max_candidate_reviews_per_business, max_final_reviews_per_business)
    rows = []
    n_candidate_kept = 0
    chunks_seen = 0
    per_biz_kept = defaultdict(int)
    progress_total = max(max_candidate_reviews, max_reviews)
    pbar = tqdm(total=progress_total, desc="Collecting capped review candidates", unit="reviews")

    for chunk in pd.read_json(review_json_path, lines=True, chunksize=chunksize):
        chunks_seen += 1
        if "business_id" not in chunk.columns or "text" not in chunk.columns:
            continue
        sub = chunk.loc[chunk["business_id"].astype(str).isin(eligible_biz_ids)].copy()
        if sub.empty:
            continue
        for col in KEEP_REVIEW_COLS:
            if col not in sub.columns:
                sub[col] = np.nan
        sub = sub[KEEP_REVIEW_COLS]
        sub["business_id"] = sub["business_id"].astype(str)
        sub["subcategories_list"] = sub["business_id"].map(biz_to_subcats)
        sub = sub.dropna(subset=["subcategories_list", "text"])
        if sub.empty:
            continue

        # Cap inside the streaming candidate pool. This avoids collecting thousands of
        # reviews from one very popular restaurant before less frequent businesses appear.
        if candidate_cap > 0:
            starting_counts = sub["business_id"].map(lambda bid: per_biz_kept.get(bid, 0)).astype(int)
            within_chunk_rank = sub.groupby("business_id").cumcount()
            keep_mask = (starting_counts + within_chunk_rank) < candidate_cap
            sub = sub.loc[keep_mask].copy()
        if sub.empty:
            continue

        for bid, count in sub["business_id"].value_counts().items():
            per_biz_kept[str(bid)] += int(count)

        rows.append(sub)
        n_candidate_kept += len(sub)
        pbar.update(min(len(sub), progress_total - pbar.n))
        if n_candidate_kept >= max_candidate_reviews:
            break
    pbar.close()

    if not rows:
        raise ValueError("No reviews collected. Check dataset paths, category gates, and MAX_REVIEWS.")

    candidates = pd.concat(rows, ignore_index=True).head(max_candidate_reviews)
    candidates["text"] = candidates["text"].astype(str)
    candidate_counts = candidates["business_id"].value_counts()

    final_pool = candidates
    if max_final_reviews_per_business > 0:
        sampled_parts = []
        for _, group in candidates.groupby("business_id", sort=False):
            n_take = min(len(group), max_final_reviews_per_business)
            sampled_parts.append(group.sample(n=n_take, random_state=RANDOM_SEED))
        final_pool = pd.concat(sampled_parts, ignore_index=True)

    if len(final_pool) > max_reviews:
        final_pool = final_pool.sample(n=max_reviews, random_state=RANDOM_SEED)
    out = final_pool.reset_index(drop=True)
    out["text"] = out["text"].astype(str)
    final_counts = out["business_id"].value_counts()

    diagnostics = {
        "sampling_strategy": "bounded_candidate_pool_with_per_business_caps",
        "random_seed": RANDOM_SEED,
        "chunks_seen": int(chunks_seen),
        "max_reviews_requested": int(max_reviews),
        "max_candidate_reviews_requested": int(max_candidate_reviews),
        "max_candidate_reviews_per_business_setting": int(candidate_cap),
        "max_final_reviews_per_business_setting": int(max_final_reviews_per_business),
        "candidate_reviews_collected": int(len(candidates)),
        "final_reviews_collected": int(len(out)),
    }
    diagnostics.update(_business_review_count_summary(candidate_counts, "candidate"))
    diagnostics.update(_business_review_count_summary(final_counts, "final"))
    diagnostics_df = pd.DataFrame([diagnostics])

    top_businesses = pd.concat(
        [
            candidate_counts.rename("candidate_review_count"),
            final_counts.rename("final_review_count"),
        ],
        axis=1,
    ).fillna(0).reset_index().rename(columns={"index": "business_id"})
    top_businesses["candidate_review_count"] = top_businesses["candidate_review_count"].astype(int)
    top_businesses["final_review_count"] = top_businesses["final_review_count"].astype(int)
    top_businesses = top_businesses.sort_values(
        ["final_review_count", "candidate_review_count"], ascending=False
    ).reset_index(drop=True)
    return out, diagnostics_df, top_businesses

### 3.3 Multi-label category corpus construction

Yelp businesses can have multiple category labels. In the main analysis, a review from an `Italian, Pizza` business contributes to both `Italian` and `Pizza`. That is faithful to Yelp's taxonomy, but it means similarity can reflect shared business/category structure as well as shared cuisine language.

The corpus builder intentionally preserves this multi-label structure for the main map. Treat this as an interpretation caveat rather than a separate overlap-diagnostic assignment.

In [ ]:
def truncate_join_text(texts, max_chars: int) -> str:
    pieces = []
    total = 0
    for t in texts:
        t = re.sub(r"\s+", " ", str(t)).strip()
        if not t:
            continue
        if total + len(t) + 1 > max_chars:
            remaining = max_chars - total
            if remaining > 100:
                pieces.append(t[:remaining])
            break
        pieces.append(t)
        total += len(t) + 1
    return " ".join(pieces)


def _review_pair_columns(reviews_df: pd.DataFrame) -> list[str]:
    base_cols = ["review_id", "business_id", "stars", "date", "text", "subcategories_list"]
    optional_cols = ["business_name", "city", "state", "raw_categories"]
    return [c for c in base_cols + optional_cols if c in reviews_df.columns]


def build_corpus_from_pairs(pairs_df: pd.DataFrame, subcategory_order: list[str], max_chars: int) -> pd.DataFrame:
    corpus_rows = []
    for subcat in subcategory_order:
        group = pairs_df.loc[pairs_df["subcategory"] == subcat]
        if group.empty:
            continue
        corpus_rows.append({
            "subcategory": subcat,
            "n_reviews": int(group["review_id"].nunique()),
            "n_businesses": int(group["business_id"].nunique()),
            "avg_review_stars": float(pd.to_numeric(group["stars"], errors="coerce").mean()),
            "text": truncate_join_text(group["text"].tolist(), max_chars),
        })
    return pd.DataFrame(corpus_rows).reset_index(drop=True)


def build_subcategory_corpus(reviews_df: pd.DataFrame, top_n: int, min_reviews: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    pairs = (
        reviews_df[_review_pair_columns(reviews_df)]
        .explode("subcategories_list", ignore_index=True)
        .rename(columns={"subcategories_list": "subcategory"})
        .dropna(subset=["subcategory", "text"])
    )
    counts = pairs["subcategory"].value_counts()
    eligible_counts = counts[counts >= min_reviews]

    focus_mode = CATEGORY_FOCUS_MODE
    if focus_mode in {"cuisine", "cuisine_focused", "food_focused"}:
        focused = [c for c in eligible_counts.index if c in CUISINE_FOCUS_SUBCATEGORIES]
        if len(focused) >= MIN_FOCUSED_SUBCATEGORIES:
            eligible = focused[:top_n]
            print(f"Using cuisine-focused subcategories ({len(eligible)} retained).")
        else:
            eligible = eligible_counts.head(top_n).index.tolist()
            print("Cuisine-focused mode did not find enough categories; using broad eligible subcategories instead.")
    else:
        eligible = eligible_counts.head(top_n).index.tolist()
        print(f"Using broad Yelp subcategory mode ({len(eligible)} retained).")

    if len(eligible) < 4:
        # For smoke tests or very small samples, relax automatically.
        eligible = counts.head(min(top_n, max(4, len(counts)))).index.tolist()
        print("Relaxed MIN_REVIEWS_PER_SUBCATEGORY because the current dataset/sample is small.")

    pairs_top = pairs.loc[pairs["subcategory"].isin(eligible)].copy()
    corpus_df = build_corpus_from_pairs(pairs_top, eligible, MAX_TEXT_CHARS_PER_SUBCATEGORY)
    corpus_df = corpus_df.sort_values("n_reviews", ascending=False).reset_index(drop=True)
    return pairs_top, corpus_df

### 3.5 Collect the capped review subset

This cell is the expensive streaming step. In real mode, it scans Yelp reviews, keeps only eligible restaurant/food businesses, and applies the per-business cap before the final sample is formed. In smoke-test mode, the same code runs against a tiny synthetic fixture.

In [ ]:
biz_to_subcats, biz_meta = collect_business_subcategories(business_path)
eligible_biz_ids = set(biz_to_subcats.keys())
reviews_df, sampling_diagnostics_df, sampling_top_businesses_df = stream_reviews_for_eligible_businesses(
    review_path,
    eligible_biz_ids,
    biz_to_subcats,
    max_reviews=MAX_REVIEWS,
    max_candidate_reviews=MAX_REVIEW_CANDIDATES,
    max_candidate_reviews_per_business=MAX_CANDIDATE_REVIEWS_PER_BUSINESS,
    max_final_reviews_per_business=MAX_FINAL_REVIEWS_PER_BUSINESS,
)

### 3.6 Attach business context and build the category corpus

Business metadata is attached now because restaurant names, cities, states, and original category strings make the saved artifacts more useful for Tasks 3-6.

In [ ]:
# Attach business metadata needed for handoff artifacts.
reviews_df["business_name"] = reviews_df["business_id"].map(lambda bid: biz_meta.get(str(bid), {}).get("name", ""))
reviews_df["city"] = reviews_df["business_id"].map(lambda bid: biz_meta.get(str(bid), {}).get("city", ""))
reviews_df["state"] = reviews_df["business_id"].map(lambda bid: biz_meta.get(str(bid), {}).get("state", ""))
reviews_df["raw_categories"] = reviews_df["business_id"].map(lambda bid: biz_meta.get(str(bid), {}).get("raw_categories", ""))

pairs_top, corpus_df = build_subcategory_corpus(reviews_df, TOP_N_SUBCATEGORIES, MIN_REVIEWS_PER_SUBCATEGORY)
selected_subcategories = corpus_df["subcategory"].tolist()

### 3.7 Inspect the subset before modeling

Before computing similarity, look for three signals: enough subcategories, enough businesses per subcategory, and no obvious concentration. If the subset is weak, changing models will not fix the core evidence problem.

In [ ]:
print("Collected review rows:", len(reviews_df))
print("Exploded review x subcategory rows:", len(pairs_top))
print("Subcategory corpus rows:", len(corpus_df))
display(corpus_df[["subcategory", "n_reviews", "n_businesses", "avg_review_stars"]].head(20))
display(sampling_diagnostics_df)

### 3.8 Save subset, sampling, and core corpus artifacts

These artifacts make Task 2 auditable and reusable by later tasks. Loading, sampling, corpus construction, and export are kept as separate functional units so the student version can replace selected solution blocks with TODOs.

In [ ]:
def save_table(df: pd.DataFrame, base_name: str):
    """Prefer Parquet when available; fall back to CSV so the notebook works in minimal environments."""
    parquet_path = OUTPUT_DIR / f"{base_name}.parquet"
    csv_path = OUTPUT_DIR / f"{base_name}.csv"
    try:
        df.to_parquet(parquet_path, index=False)
        print("Saved:", parquet_path)
    except Exception as e:
        df.to_csv(csv_path, index=False)
        print("Parquet unavailable; saved CSV instead:", csv_path)
        print("Reason:", repr(e))

save_table(reviews_df, "task2_reviews_subset")
save_table(pairs_top, "task2_review_subcategory_pairs")
save_table(corpus_df, "task2_subcategory_corpus")
sampling_diagnostics_df.to_csv(OUTPUT_DIR / "task2_sampling_diagnostics.csv", index=False)
sampling_top_businesses_df.to_csv(OUTPUT_DIR / "task2_sampling_top_businesses.csv", index=False)

print("Saved sampling diagnostics and core subset artifacts.")


### Student checkpoint: subset quality

Before modeling, inspect whether the subset is broad enough and whether one business dominates the evidence. If the subset is weak, a different representation will not rescue the map.


In [ ]:
# TODO (Category E — Guided inspection): Report key statistics about the corpus subset.
# Estimated time: ~0.5 h
#
# Expected inputs:
#   corpus_df               : DataFrame with columns subcategory, n_reviews, n_businesses, avg_review_stars
#   pairs_top               : long-form DataFrame with columns review_id, subcategory, text
#   sampling_diagnostics_df : DataFrame from the capped sampler
#   sampling_top_businesses_df : DataFrame of top businesses by review count
#
# Expected output (print statements + displays, no files):
#   - Number of subcategories in corpus_df
#   - Total unique reviews in pairs_top
#   - Top 15 subcategories by corpus_df displayed
#   - sampling_diagnostics_df displayed
#   - Top 10 businesses from sampling_top_businesses_df displayed
#   - Average number of subcategory labels per review (pairs_top groupby review_id)
print("Number of subcategories used:", len(corpus_df))
# TODO: print total unique reviews using pairs_top["review_id"].nunique()
# TODO: display top 15 rows of corpus_df[["subcategory", "n_reviews", "n_businesses", "avg_review_stars"]]
# TODO: display sampling_diagnostics_df
# TODO: display sampling_top_businesses_df.head(10)
# TODO: compute and print mean_labels — average number of subcategory labels per unique review_id
#       Hint: pairs_top.groupby("review_id")["subcategory"].nunique().mean()
# my code
print("Total unique reviews in pairs_top:", pairs_top["review_id"].nunique())

print("Top 15 subcategories by number of reviews:")
display(
    corpus_df[
        ["subcategory", "n_reviews", "n_businesses", "avg_review_stars"]
    ].head(15)
)

print("Sampling diagnostics:")
display(sampling_diagnostics_df)

print("Top 10 businesses by review count:")
display(sampling_top_businesses_df.head(10))

mean_labels = pairs_top.groupby("review_id")["subcategory"].nunique().mean()

print("Average number of subcategory labels per review:", round(mean_labels, 3))

## Data Mining Concept: Text Similarity as a Representation Choice — What TF-IDF and Embeddings Each Measure

**The text-mining question:** Which cuisines or restaurant categories are similar to each other based on their review text?

Task 2 keeps the comparison deliberately focused:

1. **TF-IDF similarity**: a classical, transparent bag-of-words representation. It highlights categories that share distinctive vocabulary.
2. **Embedding similarity**: a modern semantic representation. It can place categories closer together even when they use different surface words.

The goal is not to try every clustering algorithm. The goal is to understand how the representation changes the map and how one clustering result should be interpreted in light of that representation.


## 4. Pipeline A — Classical TF-IDF similarity

We build one classical text representation: **TF-IDF**. It is transparent, inspectable, and still a strong baseline for text mining. The resulting similarity matrix gives the lexical cuisine/subcategory map that will be compared against the modern embedding map.


### 4.1 TF-IDF vectorizer and heatmap helpers

TF-IDF is the classical baseline for Task 2. It gives students inspectable top terms and a lexical similarity matrix before moving to embedding-based similarity.


## About TF-IDF Vectorization

TF-IDF vectorization converts text into weighted term features that increase the importance of terms that are common within a document but relatively uncommon across the corpus. In Task 2, it strengthens the classical cuisine/subcategory similarity pipeline by reducing the influence of generic Yelp vocabulary and emphasizing terms that distinguish one category from another. This is a central data mining representation choice: the same reviews can produce a different cuisine map simply because the feature weighting changes.

Students use TF-IDF to build cosine similarity matrices, top-term tables and clustering inputs. Because TF-IDF remains sparse and inspectable, it also supports direct interpretation: students can explain why two categories are close by looking at their influential terms. Compared with sentence-transformer embeddings, TF-IDF is less semantically flexible but often easier to audit and more stable in constrained smoke-test environments.

**Key Resources:**
- [scikit-learn TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) - Official API for TF-IDF feature extraction.
- [scikit-learn Text Feature Extraction](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction) - Guide to count and TF-IDF representations.
- [Wikipedia: TF-IDF](https://en.wikipedia.org/wiki/Tf%E2%80%93idf) - Conceptual overview of term-frequency and inverse-document-frequency weighting.

Why use this method?
- Produces interpretable distinctive-term evidence.
- Offers a stronger classical baseline than raw counts.
- Supports similarity, clustering, and top-term diagnostics.

In [ ]:
# TODO (Category B — Classic data mining): Implement TF-IDF vectorization and similarity helpers.
# Estimated time: ~2 h
#
# Implement the following functions used in Pipeline A and B:
#
# ── A) vectorize_corpus(corpus_df, use_idf, extra_stop_words=None) ────────────
#   Returns: (vectorizer, X)  where X is a sparse matrix, shape (n_subcategories, vocab)
#   - If use_idf=True: standard TF-IDF with sublinear_tf=True, norm="l2"
#   - If use_idf=False: term frequency only (set use_idf=False in TfidfVectorizer)
#   - Use MAX_FEATURES, min_df=1, max_df=0.95, NGRAM_RANGE from config
#   - If extra_stop_words given: extend sklearn ENGLISH_STOP_WORDS
#   - Fit on corpus_df["text"].fillna("")
#
# ── B) hierarchical_order_from_similarity(S) ─────────────────────────────────
#   Returns: list[int]  — permutation of row/column indices that groups similar
#   rows together (for ordered heatmaps).
#   Use scipy.cluster.hierarchy linkage on 1-S (distance), then leaves ordering.
#   Return list(range(len(S))) as fallback if scipy is unavailable or S is tiny.
#
# ── C) save_similarity_matrix(S, labels, filename) ───────────────────────────
#   Save S as a DataFrame with index=labels, columns=labels to OUTPUT_DIR/filename.
#
# ── D) plot_similarity_heatmap(S, labels, title, filename, order=None) ───────
#   Plot a square heatmap of S (reordered by order if given) and save to OUTPUT_DIR.
#   Use seaborn.heatmap or matplotlib imshow with a diverging colormap.
#   Add subcategory labels on both axes (rotate x-axis labels 45°).
#
# ── E) top_terms_for_rows(X, vectorizer, row_labels, n=10) ───────────────────
#   Returns: DataFrame with columns [subcategory, top_terms] where top_terms is
#   a space-separated string of the top-n TF-IDF terms for each row.
#   Hint: argsort each row descending, look up terms with vectorizer.get_feature_names_out()

def vectorize_corpus(corpus_df, use_idf: bool, extra_stop_words=None):
    # TODO: build TfidfVectorizer with the parameters described above
    # TODO: fit_transform corpus_df["text"].fillna("") → X
    # TODO: return (vectorizer, X)
    # my code
    # utilize the default stop words from sklearn
    default_stop_words=set(ENGLISH_STOP_WORDS)
    # if users provide extra stop words, update the default stop words
    if extra_stop_words is not None:
        default_stop_words.update(set(extra_stop_words))

    # create the vectorizer
    vectorizer=TfidfVectorizer(
        # set the parameter for the max features
        max_features=MAX_FEATURES,
        # set the parameter for the min and max document frequency
        min_df=1,
        max_df=0.95,
        # set the parameter for the ngram range
        ngram_range=NGRAM_RANGE,
        # set the parameter for the stop words
        stop_words=list(default_stop_words),
        # set the parameter for the use of idf
        use_idf=use_idf,
        # set the parameter for the sublinear tf scaling
        sublinear_tf=use_idf,
        # set the parameter for the norm
        norm="l2",
    )
    # get the text data
    texts=corpus_df["text"].fillna("")
    # fit the vectorizer
    X=vectorizer.fit_transform(texts)
    # return the fitted vectorizer and the document-term matrix
    return vectorizer, X
        

def hierarchical_order_from_similarity(S: np.ndarray) -> list:
    if len(S) <= 2:
        return list(range(len(S)))
    # TODO: compute linkage on 1-S (squareform), return dendrogram leaves ordering
    # Fallback to list(range(len(S))) on any exception
    # my code
    # attempt to use scipy to sort the similarity matrix
    try:
        from scipy.cluster.hierarchy import linkage, leaves_list
        from scipy.spatial.distance import squareform
        # convert the similarity matrix to a distance matrix
        distance_matrix=1-S
        # avoid the negative distance or the abnormal value
        distance_matrix=np.clip(distance_matrix, 0, 2)
        # set the diagnal of the distance matrix to 0
        np.fill_diagonal(distance_matrix, 0)
        # convert the squareform distance matrix to a condensed distance matrix
        condensed_distance_matrix=squareform(distance_matrix,checks=False)
        # use the average linkage method to compute the linkage
        linked=linkage(condensed_distance_matrix,method="average")
        # get the leaves ordering   
        order=leaves_list(linked)   
        # return the leaves ordering
        return order.tolist()
    except Exception:
        return list(range(len(S)))
        

def save_similarity_matrix(S: np.ndarray, labels: list, filename: str):
    # TODO: save pd.DataFrame(S, index=labels, columns=labels) to OUTPUT_DIR / filename
    # my code
    # create a dataframe from the similarity matrix
    similarity_df=pd.DataFrame(S,index=labels,columns=labels)
    # save the dataframe to the output directory
    similarity_df.to_csv(OUTPUT_DIR/filename,index=True,header=True)

def plot_similarity_heatmap(S: np.ndarray, labels: list, title: str, filename: str, order=None):
    # TODO: optionally reorder S by order; plot heatmap with labeled axes; save figure
    # my code
    # if the order is provided, reorder the similarity matrix
    if order is not None:
        S_plot=S[np.ix_(order,order)]
        # sort the labels
        labels_plot=[labels[i] for i in order]
    else:
        S_plot=S
        labels_plot=labels
    # create a square figure
    plt.figure(figsize=(10,10))

    # use imshow to plot the heatmap
    plt.imshow(S_plot,cmap="coolwarm",vmin=0,vmax=1)
    # set the color bar
    plt.colorbar(label="Cosine Similarity")
    # set the x axis labels
    plt.xticks(
        ticks=range(len(labels_plot)),
        labels=labels_plot,
        rotation=45,
        ha="right",
    )
    # set the y axis labels
    plt.yticks(
        ticks=range(len(labels_plot)),
        labels=labels_plot,
    )
    # set the title of the heatmap
    plt.title(title)
    # automatically adjust the subplot parameters
    plt.tight_layout()
    # save the heatmap
    plt.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches="tight")
    # show the heatmap
    plt.show()
    # close the plot
    plt.close()

def top_terms_for_rows(X, vectorizer, row_labels: list, n: int = 10):
    # TODO: for each row in X, find the top-n term indices by TF-IDF score
    # TODO: return DataFrame[subcategory, top_terms] where top_terms is space-separated
    # my code
    # get all the feature names
    feature_names=vectorizer.get_feature_names_out()
    # create a list to store the top terms
    top_terms_list=[]
    # iterate each row in row_labels
    for row_index,label in enumerate(row_labels):
        # get the current vector of the row
        row=X[row_index]
        # convert the row to a numpy array
        row_scores=row.toarray().ravel()
        # sort the row scores in descending order and get the top-n indices
        top_n_indices=row_scores.argsort()[::-1][:n]
        # keep the top-n terms
        top_terms=[
            feature_names[i]
            for i in top_n_indices
            if row_scores[i] > 0
        ]
        # add the top terms to the list
        top_terms_list.append(
            {
                "subcategory":label,
                "top_terms":" ".join(top_terms)
            }
        )
    # create a dataframe from the top terms
    top_terms_df=pd.DataFrame(top_terms_list)
    # return the dataframe
    return top_terms_df
    

## About Cosine Similarity Matrix

Cosine similarity measures the angle between two vectors rather than their raw magnitude. In Task 2, each cuisine or subcategory becomes a vector under a representation such as TF-IDF or embeddings, and cosine similarity turns those vectors into a pairwise category map. This lets students compare representations while holding the similarity measure constant.

The method is especially useful for review text because categories can have different review volumes. A large category should not automatically look close to every other category simply because it contains more words or examples. Cosine similarity partially controls that by focusing on direction in feature space. Students can then inspect heatmaps and similarity matrices to decide whether observed neighborhoods are meaningful, driven by Yelp multi-label structure, sparse text, or representation design.

In the classical-vs-modern comparison, cosine similarity acts as a shared measuring instrument: the vectors change, but the geometric comparison remains consistent. That makes disagreements across methods easier to interpret.

**Key Resources:**
- [scikit-learn cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) - Official API reference.
- [Wikipedia: Cosine Similarity](https://en.wikipedia.org/wiki/Cosine_similarity) - Conceptual description and formula.
- [scikit-learn Pairwise Metrics](https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity) - Documentation for pairwise similarity tools.

Why use this method?
- Normalizes comparison across differently sized category documents.
- Works with sparse lexical vectors and dense embeddings.
- Produces readable heatmaps and pairwise similarity evidence.

In [ ]:
# TODO (Category E — Guided sklearn): Run Pipeline A — TF-IDF similarity.
# Estimated time: ~0.5 h
#
# Expected inputs:
#   corpus_df        : DataFrame with "subcategory" and "text" columns
#   vectorize_corpus : function you implemented above
#   OUTPUT_DIR       : Path for saving artifacts
#
# Steps:
# 1. subcategories = corpus_df["subcategory"].tolist()
# 2. Call vectorize_corpus(corpus_df, use_idf=True) → vec_tfidf, X_tfidf
# 3. Compute S_tfidf = cosine_similarity(X_tfidf)  (shape: n_subcats × n_subcats)
# 4. Get order_tfidf = hierarchical_order_from_similarity(S_tfidf)
# 5. Save: save_similarity_matrix(S_tfidf, subcategories, "similarity_tfidf_matrix.csv")
# 6. Plot: plot_similarity_heatmap(S_tfidf, subcategories, "...", "similarity_tfidf_heatmap.png", order_tfidf)
#
# Required output variables for downstream cells:
#   subcategories, vec_tfidf, X_tfidf, S_tfidf, order_tfidf

# TODO: set subcategories
subcategories = corpus_df["subcategory"].tolist()

# TODO: vectorize and compute TF-IDF cosine similarity
# vectorize the corpus by using the TF-IDF
vec_tfidf, X_tfidf = vectorize_corpus(corpus_df, use_idf=True)
# compute the cosine similarity
S_tfidf = cosine_similarity(X_tfidf)
# get the hierarchical order
order_tfidf = hierarchical_order_from_similarity(S_tfidf)

# TODO: save and plot
# save the similarity matrix
save_similarity_matrix(
    S_tfidf, 
    subcategories, 
    "similarity_tfidf_matrix.csv")
# plot the heatmap
plot_similarity_heatmap(
    S_tfidf, 
    subcategories, 
    "Subcategory similarity — TF-IDF"   , 
    "similarity_tfidf_heatmap.png", 
    order_tfidf
    )

# --- Validation ---
assert S_tfidf is not None and S_tfidf.shape == (len(subcategories), len(subcategories))
assert (OUTPUT_DIR / "similarity_tfidf_matrix.csv").exists()
print(f"\u2713 TF-IDF similarity matrix: {S_tfidf.shape}")


In [ ]:
top_terms_df = top_terms_for_rows(X_tfidf, vec_tfidf, subcategories, n=TOP_TERMS_PER_SUBCATEGORY)
top_terms_df.to_csv(OUTPUT_DIR / "task2_top_tfidf_terms_by_subcategory.csv", index=False)
print("Top TF-IDF terms by subcategory:")
display(top_terms_df.head(20))

## 5. Pipeline B — Modern embedding-based similarity

The preferred modern route embeds sampled Yelp review snippets with a pretrained sentence model and averages them into one vector per subcategory. This produces a dense semantic representation that can connect categories even when they use different surface vocabulary.

In real assignment modes, the sentence-transformer path is required by default. If `sentence-transformers` or model loading is unavailable in smoke-test mode, the notebook may fall back to TF-IDF + SVD so smoke-test validation can still check the rest of the notebook. That fallback must be labeled honestly as a fallback, not as equivalent to pretrained sentence embeddings.

The comparison is now intentionally simple: TF-IDF gives the classical lexical map, embeddings give the modern semantic map, and a single clustering result summarizes structure in the embedding space.


### 5.1 Snippet sampling and sentence-transformer embeddings

Instead of embedding an enormous concatenated category document, we sample bounded review snippets from each subcategory and average their normalized sentence embeddings. This keeps runtime bounded and makes the representation depend on Yelp review text, not just category labels.

## About Sentence-Transformer Embeddings

Sentence-transformer embeddings map review text into dense vector spaces where geometric distance approximates semantic similarity. In Task 2, these embeddings provide the representational backbone for modern cuisine and subcategory similarity, allowing the pipeline to group restaurant categories by meaning rather than exact term overlap. This is especially useful when comparing cuisines whose reviews share common vocabulary but describe distinct dining experiences.

Pedagogically, this method gives you a concrete contrast with classical bag-of-words features. Count-based representations are transparent but sparse and lexically rigid, while sentence embeddings are semantically expressive but less directly interpretable. By comparing cuisine clusters and similarity maps produced by different representations, you learn that representation choice controls which culinary groupings become visible.

In Task 2, embeddings are compared directly against TF-IDF in the cuisine similarity pipeline. This comparison is central to the task: you must assess whether geometric meaning-similarity produces a more useful cuisine map than a classical sparse representation. Even when a full sentence-transformer is unavailable, understanding the embedding concept guides fallback designs and interpretation standards. In this course framing, embeddings are not treated as automatic upgrades; instead, they are evaluated against classical baselines on interpretability, stability, and computational practicality.

**Key Resources:**
- [Sentence-Transformers Documentation](https://www.sbert.net/) - Official documentation and model usage patterns.
- [Sentence-BERT Paper](https://aclanthology.org/D19-1410/) - Foundational architecture for semantic sentence embeddings.
- [Hugging Face Sentence-Transformers Models](https://huggingface.co/sentence-transformers) - Model catalog and deployment references.

In [ ]:
# TODO (Category A — Modern library): Build per-subcategory embedding vectors.
# Estimated time: ~2.5 h
#
# Implement TWO functions:
#
# ── A) sample_snippets_by_subcategory(pairs_df, max_snippets, char_limit) ────
#   Returns: dict[subcategory_name → list[str]]  (truncated review text snippets)
#   - For each subcategory in pairs_df, deduplicate by review_id.
#   - Randomly sample up to max_snippets snippets (use np.random.default_rng(RANDOM_SEED)).
#   - Truncate each snippet to char_limit characters; strip whitespace.
#
# ── B) build_embedding_vectors(pairs_df, corpus_df) ─────────────────────────
#   Returns: (X, method_label)
#     X           : 2D numpy array, shape (n_subcategories, embedding_dim), L2-normalized
#     method_label: descriptive string of the method used
#
#   Primary path — sentence transformers:
#     - Load SentenceTransformer(SENTENCE_TRANSFORMER_MODEL) with optional cache.
#     - For each subcategory in corpus_df["subcategory"]:
#         snippets = sample_snippets_by_subcategory result for that subcategory
#         encode all snippets (batch_size=32, normalize_embeddings=True)
#         average the embeddings → one vector per subcategory
#     - Stack all vectors, L2-normalize the matrix (sklearn normalize).
#     - Return (X, f"{SENTENCE_TRANSFORMER_MODEL} averaged review snippets").
#
#   Fallback path (if SentenceTransformer import fails or REQUIRE_SENTENCE_TRANSFORMER=False):
#     - Use TruncatedSVD on X_tfidf to produce dense embeddings.
#     - n_components = min(50, X_tfidf.shape[0]-1, X_tfidf.shape[1]-1)
#     - L2-normalize the result.
#     - Return (X_svd, "fallback: TF-IDF + TruncatedSVD(n_components)").
#
# Hint: from sentence_transformers import SentenceTransformer; model.encode(snippets) → array
#       sklearn.preprocessing.normalize(X) normalizes rows to unit length.

def sample_snippets_by_subcategory(pairs_df, max_snippets_per_subcategory: int, char_limit: int) -> dict:
    # TODO: for each subcategory, deduplicate → sample → truncate → return dict
    # my code
    # create a random number generator to ensure the reproducibility of the sampling results
    random_number_generator=np.random.default_rng(RANDOM_SEED)

    # create a dictionary to store the snippets for each subcategory
    snippets_by_cat={}

    # iterate each subcategory in the pairs_df
    for subcategory, group_df in pairs_df.groupby("subcategory"):

        # deduplicate the reviews in the same subcategory by review_id
        unique_reviews = group_df.drop_duplicates(subset="review_id").copy()
        # get the text column and fill the missing values with empty strings
        text_series = unique_reviews["text"].fillna("").astype(str)
        # remove the whitespace at the beginning and end of each text
        text_series = text_series.str.strip()
        # remove the empty texts
        text_series = text_series[text_series != ""]

        # convert the pandas Series to a Python list
        snippets = text_series.tolist()

        # if the number of snippets exceeds the upper limit, randomly sample
        if len(snippets) > max_snippets_per_subcategory:

            # randomly sample the snippets without replacement
            selected_indices = random_number_generator.choice(len(snippets), size=max_snippets_per_subcategory, replace=False)
            # get the snippets by the selected indices
            snippets = [snippets[i] for i in selected_indices]

        # truncate the snippets to the char_limit
        truncated_snippets = [text[:char_limit].strip() for text in snippets]

        # store the snippets for the current subcategory in the dictionary
        snippets_by_cat[subcategory] = truncated_snippets

    # return the dictionary of subcategory to snippets
    return snippets_by_cat

def build_embedding_vectors(pairs_df, corpus_df):
    labels = corpus_df["subcategory"].tolist()
    snippets_by_cat = sample_snippets_by_subcategory(pairs_df, MAX_SNIPPETS_PER_SUBCATEGORY, SNIPPET_CHAR_LIMIT)
    try:
        from sentence_transformers import SentenceTransformer
        # TODO: load model; encode snippets per subcategory; average; normalize; return (X, label)
        # my code
        if not REQUIRE_SENTENCE_TRANSFORMER:
            raise ImportError("REQUIRE_SENTENCE_TRANSFORMER=False; forcing fallback.")

        # create a dictionary to store the optional parameters for loading the model
        model_kwargs = {}
        # if the model cache directory is set, add the cache_folder parameter
        if MODEL_CACHE_DIR:
            # set the cache folder for the sentence-transformer
            model_kwargs["cache_folder"] = MODEL_CACHE_DIR
        # if the local model files are required, add the local_files_only parameter
        if LOCAL_FILES_ONLY:
            # set the local_files_only parameter to True
            model_kwargs["local_files_only"] = True

        # load the sentence-transformer model
        model = SentenceTransformer(SENTENCE_TRANSFORMER_MODEL, **model_kwargs)

        # create an empty list to store the embeddings 
        embedding_rows = []

        # iterate each subcategory in the corpus_df
        for label in labels:

            # get the snippets for the current subcategory
            snippets = snippets_by_cat.get(label, [])

            # if the current subcategory has no snippets, use the aggregated text in corpus_df as a fallback
            if len(snippets) == 0:
                fallback_text = (
                    corpus_df.loc[
                        corpus_df["subcategory"] == label,
                        "text"
                    ]
                    .fillna("")
                    .iloc[0]
                )
                snippets = [fallback_text]

            # encode the snippets into embeddings by using the sentence-transformer
            snippet_embeddings = model.encode(snippets, batch_size=32, normalize_embeddings=True)

            # convert the encoding results to a numpy array
            snippet_embeddings = np.asarray(snippet_embeddings)

            # average the embeddings for multiple snippets in the same subcategory
            category_embedding = snippet_embeddings.mean(axis=0)

            # add the average embedding for the current subcategory to the list
            embedding_rows.append(category_embedding)

        # stack all the embeddings for all subcategories into a 2D matrix
        X = np.vstack(embedding_rows)

        # normalize each row of the embedding matrix
        X = normalize(X)

        # create the method description
        method_label = f"{SENTENCE_TRANSFORMER_MODEL} averaged review snippets"

        # return the embedding matrix and the method description
        return X, method_label
    except Exception as e:
        if REQUIRE_SENTENCE_TRANSFORMER:
            raise RuntimeError(
                "Sentence-transformer embeddings are required for real Task 2 runs. "
                "Use smoke_test mode for fallback-only validation."
            ) from e
        print("Embedding model unavailable; using TF-IDF + SVD fallback.")
        print("Reason:", repr(e))
        # TODO: apply TruncatedSVD to X_tfidf; normalize; return (X_svd, fallback_label)
        # my code
        # calculate the number of dimensions for TruncatedSVD
        n_components = min(50, X_tfidf.shape[0] - 1, X_tfidf.shape[1] - 1)

        # create the TruncatedSVD model
        svd_model = TruncatedSVD(n_components=n_components, random_state=RANDOM_SEED)

        # use SVD to compress the TF-IDF sparse matrix into dense embeddings
        X_svd = svd_model.fit_transform(X_tfidf)

        # normalize the SVD embeddings
        X_svd = normalize(X_svd)

        # create the fallback method description
        fallback_label = f"fallback: TF-IDF + TruncatedSVD({n_components})"

        # return the fallback embedding and the method description
        return X_svd, fallback_label


### 5.2 Two-dimensional layouts

The 2-D map is a communication tool, not the source of truth. UMAP is used when available; PCA is a deterministic fallback for small or constrained runs. You should interpret layouts alongside the similarity heatmaps and HDBSCAN cluster assignments.

## About UMAP 2-D Projection

UMAP is a nonlinear dimensionality-reduction method that projects high-dimensional vectors into a lower-dimensional layout while attempting to preserve local neighborhood structure. In Task 2, students use UMAP or a fallback projection to create a communication-oriented 2-D map of cuisine or subcategory embeddings. The map helps students inspect which categories appear near one another in semantic space, but it should be treated as visual evidence rather than ground truth.

This method is pedagogically valuable because projection choices can change what the viewer perceives. A 2-D map compresses information, so students must compare it with similarity matrices and HDBSCAN cluster assignments before making claims. When UMAP is unavailable, PCA or another deterministic fallback keeps the notebook runnable while preserving the lesson about representation geometry.

In the course arc, UMAP introduces modern visualization practice that recurs in topic modeling, embedding clustering, and temporal drift analysis.

**Key Resources:**
- [UMAP Documentation](https://umap-learn.readthedocs.io/en/latest/) - Official guide to UMAP concepts and parameters.
- [UMAP Paper](https://arxiv.org/abs/1802.03426) - Foundational paper describing the method.
- [scikit-learn Manifold Learning](https://scikit-learn.org/stable/modules/manifold.html) - Broader documentation on projection and manifold methods.

Why use this method?
- Creates readable category maps from dense embeddings.
- Supports visual comparison with clustering labels.
- Teaches caution about projection artifacts.

In [ ]:
# TODO (Category A — Modern library): Implement UMAP 2-D layout with PCA fallback.
# Estimated time: ~2 h
#
# ── A) make_2d_layout(X, method_label) ───────────────────────────────────────
#   Returns: (coords, layout_label)
#     coords       : 2D numpy array, shape (n_subcategories, 2)
#     layout_label : "UMAP" or "PCA fallback"
#
#   Primary: import umap; UMAP(n_components=2, metric="cosine", random_state=RANDOM_SEED,
#            n_neighbors=min(10, max(2, len(X)-1))).fit_transform(X)
#   Fallback: PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X)
#   Use try/except around the UMAP import and fit.
#
# ── B) plot_2d_map(coords, labels, title, filename, cluster_labels=None) ─────
#   Scatter plot: coords[:, 0] vs coords[:, 1].
#   If cluster_labels given: color points by cluster (use a qualitative colormap; label -1 as "noise").
#   Annotate each point with its subcategory label (use ax.annotate).
#   Save to OUTPUT_DIR / filename.

def make_2d_layout(X: np.ndarray, method_label: str):
    try:
        import umap
        # TODO: fit UMAP and return (coords, "UMAP")
        # 计算 UMAP 的邻居数量，不能超过样本数减 1
        n_neighbors = min(10, max(2, len(X) - 1))

        # 创建 UMAP 模型
        reducer = umap.UMAP(
            n_components=2,
            metric="cosine",
            random_state=RANDOM_SEED,
            n_neighbors=n_neighbors
        )

        # 用 UMAP 把 X 降维到二维坐标
        coords = reducer.fit_transform(X)

        # 返回二维坐标和方法名称
        return coords, "UMAP"
    except Exception:
        # TODO: fit PCA and return (coords, "PCA fallback")
        # 创建 PCA 模型，把数据降到二维
        pca = PCA(
            n_components=2,
            random_state=RANDOM_SEED
        )

        # 用 PCA 把 X 降维到二维坐标
        coords = pca.fit_transform(X)

        # 返回二维坐标和 fallback 方法名称
        return coords, "PCA fallback"

def plot_2d_map(coords: np.ndarray, labels: list, title: str, filename: str, cluster_labels=None):
    # TODO: scatter plot with optional cluster coloring; annotate subcategory labels; save figure
    # 创建画布和坐标轴
    fig, ax = plt.subplots(figsize=(10, 8))

    # 如果没有提供 cluster labels，就用同一种颜色画所有点
    if cluster_labels is None:

        # 画普通散点图
        ax.scatter(
            coords[:, 0],
            coords[:, 1],
            s=80,
            alpha=0.8
        )

    # 如果提供了 cluster labels，就根据 cluster 给点上色
    else:

        # 把 cluster_labels 转成 numpy array
        cluster_labels_array = np.array(cluster_labels)

        # 找出所有不同的 cluster label
        unique_clusters = sorted(set(cluster_labels_array.tolist()))

        # 创建一个颜色映射，使用 tab10 这种分类颜色
        cmap = plt.get_cmap("tab10")

        # 遍历每一个 cluster
        for color_index, cluster_id in enumerate(unique_clusters):

            # 找出属于当前 cluster 的点
            mask = cluster_labels_array == cluster_id

            # 如果 cluster_id 是 -1，就把它标记为 noise
            if cluster_id == -1:

                # 设置图例名字为 noise
                legend_label = "noise"

                # noise 用灰色显示
                color = "lightgray"

            # 如果不是 -1，就正常显示 cluster 编号
            else:

                # 设置图例名字为 cluster 编号
                legend_label = f"cluster {cluster_id}"

                # 从 colormap 中取一个颜色
                color = cmap(color_index % 10)

            # 画当前 cluster 的散点
            ax.scatter(
                coords[mask, 0],
                coords[mask, 1],
                s=80,
                alpha=0.85,
                color=color,
                label=legend_label
            )

        # 显示图例
        ax.legend()

    # 遍历每个点，给它加上 subcategory 名字
    for i, label in enumerate(labels):

        # 在点的旁边标注文字
        ax.annotate(
            label,
            (coords[i, 0], coords[i, 1]),
            fontsize=9,
            xytext=(4, 4),
            textcoords="offset points"
        )

    # 设置图片标题
    ax.set_title(title)

    # 设置 x 轴名称
    ax.set_xlabel("Dimension 1")

    # 设置 y 轴名称
    ax.set_ylabel("Dimension 2")

    # 添加浅色网格，方便看位置
    ax.grid(True, alpha=0.25)

    # 自动调整布局，避免文字被截断
    plt.tight_layout()

    # 保存图片到 OUTPUT_DIR
    plt.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches="tight")

    # 显示图片
    plt.show()


In [ ]:
# TODO (Category E — Guided): Run Pipeline B — build embedding vectors and compute similarity.
# Estimated time: ~0.5 h
#
# Steps:
# 1. X_emb_raw, embedding_method = build_embedding_vectors(pairs_top, corpus_df)
# 2. X_emb = X_emb_raw  (keep for downstream compatibility)
# 3. S_emb_raw = cosine_similarity(X_emb_raw)  → shape (n_subcats, n_subcats)
# 4. S_emb = S_emb_raw; order_emb = hierarchical_order_from_similarity(S_emb_raw)
# 5. save_similarity_matrix(S_emb, subcategories, "similarity_embedding_matrix.csv")
# 6. Print: embedding_method, S_emb.min(), S_emb.max()
#
# Required output variables: X_emb_raw, X_emb, embedding_method, S_emb_raw, S_emb, order_emb

# TODO: build embedding vectors
X_emb_raw, embedding_method = build_embedding_vectors(pairs_top, corpus_df)

X_emb = X_emb_raw
embedding_cluster_method = embedding_method

# TODO: compute cosine similarity and hierarchical order
S_emb_raw = cosine_similarity(X_emb_raw)
S_emb = S_emb_raw
order_emb_raw = hierarchical_order_from_similarity(S_emb_raw)
order_emb = order_emb_raw

# TODO: save similarity matrix; print diagnostics
# 把 embedding 相似度矩阵保存成 CSV 文件
save_similarity_matrix(
    S_emb,
    subcategories,
    "similarity_embedding_matrix.csv"
)
# --- Validation ---
assert X_emb is not None, "build_embedding_vectors returned None"
assert S_emb is not None and S_emb.shape == (len(subcategories), len(subcategories))
assert (OUTPUT_DIR / "similarity_embedding_matrix.csv").exists()
print(f"\u2713 Embedding similarity matrix: {S_emb.shape}, method: {embedding_method}")


### 5.3 Visualize embedding similarity and layout

Use these figures as evidence for the classical-vs-modern comparison. Compare the structure of
TF-IDF and embedding heatmaps and interpret any notable agreements or disagreements.

In [ ]:
plot_similarity_heatmap(
    S_emb, subcategories,
    "Subcategory similarity — embeddings",
    "similarity_embedding_heatmap.png",
    order=order_emb,
)
coords_emb, layout_method = make_2d_layout(X_emb, embedding_cluster_method)
plot_2d_map(
    coords_emb, subcategories,
    f"2-D subcategory layout — embeddings ({layout_method})",
    "layout_embedding.png",
)

## Data Mining Concept: Clustering as Unsupervised Structure Discovery

**The unsupervised learning question:**

Given a **similarity space** (TF-IDF vectors or sentence-transformer embeddings),
can we discover **natural groupings** of cuisines without pre-defined labels?
Are there 3 clusters? 5? 10? Which cuisines belong together?

This is a **clustering problem** — a form of unsupervised structure discovery.
There are no ground-truth cluster labels; the algorithm must infer structure from
the data’s geometry.

### HDBSCAN — The Density Paradigm

**Core idea**: A cluster is a **dense region of the feature space** separated from
other dense regions by sparser areas. HDBSCAN (Hierarchical Density-Based Spatial
Clustering of Applications with Noise) detects arbitrarily shaped clusters and
assigns noise points to a special “no-cluster” label (-1) rather than
forcing them into a cluster.

**What this paradigm assumes:**
- Clusters are **regions of high density** separated by low-density gaps.
- The **number of clusters is inferred** from the data, not specified up front.
- Noise and outliers are **explicitly identified** rather than folded into the
  nearest cluster.

**Why HDBSCAN is useful for cuisine/category mapping:**
Cuisine groups in embedding space often have irregular shapes and uneven densities.
HDBSCAN handles these cases better than fixed-K methods and produces a noise class
that highlights genuinely ambiguous cuisines — a useful finding in itself.

**Key limitation:** HDBSCAN’s output can be sensitive to `min_cluster_size`
and `min_samples` settings. Interpret the cluster count as a diagnostic signal,
not a definitive answer.


## 6. Clustering

We use **HDBSCAN**, a density-based clustering method that discovers groups of varying sizes
and identifies noise points (label −1) without requiring a pre-specified cluster count.

HDBSCAN may label some subcategories as noise. That is a valid result worth discussing: it
indicates that those categories do not form a dense neighbourhood under this representation.
Compare the HDBSCAN result with the similarity heatmap to explain which cuisines cluster
naturally and which are genuine outliers.

### 6.1 Clustering helper functions

Task 2 uses one clustering method for the final map. HDBSCAN is used when available because it does not require choosing a fixed number of clusters and can mark weakly supported points as noise. The fallback behavior keeps smoke-test runs stable.


## About HDBSCAN Density Clustering

HDBSCAN is a density-based clustering method that groups observations by dense regions and can label weakly supported points as noise. In Task 2, it serves as a modern diagnostic for whether cuisine or subcategory embeddings form stable dense groups without forcing a fixed number of clusters. This makes it a useful contrast with fixed-K methods such as KMeans, which assign every category to one of a preset number of clusters.

For Yelp category maps, HDBSCAN can reveal that some apparent neighborhoods are strong while others are too sparse or ambiguous to cluster confidently. You should interpret noise labels and small clusters as evidence, not as failures. The method also reinforces a key lesson from modern embedding workflows: representation quality, sample size, and density assumptions interact.

Because HDBSCAN may be unavailable in some Colab or smoke-test environments, the notebook can use DBSCAN or a clearly labeled fallback. The pedagogical goal remains the same: use HDBSCAN to surface whether the embedding space has stable cluster structure, and interpret what noise points reveal about the data.

**Key Resources:**
- [HDBSCAN Documentation](https://hdbscan.readthedocs.io/en/latest/) - Official documentation and examples.
- [scikit-learn DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html) - Related density-clustering fallback reference.
- [McInnes et al., HDBSCAN paper](https://joss.theoj.org/papers/10.21105/joss.00205) - Software paper describing the implementation.

Why use this method?
- Avoids forcing every category into a fixed cluster count.
- Surfaces ambiguity through noise/outlier labels.
- Surfaces whether the embedding space has stable dense structure without imposing a fixed K.

In [ ]:
# TODO (Category A — Modern library): Implement HDBSCAN density clustering helpers.
# Estimated time: ~2.5 h
#
# Implement THREE functions:
#
# ── A) cluster_quality(X, cluster_labels, metric="cosine") ───────────────────
#   Returns: float (silhouette score) or np.nan if <2 clusters or too few points.
#   - Exclude noise points (label == -1) from the silhouette computation.
#   - Use sklearn silhouette_score; wrap in try/except.
#
# ── B) run_hdbscan_diagnostic(X, labels) ─────────────────────────────────────
#   Returns: (cluster_assignments, method_label_string)
#   - Try: import hdbscan; HDBSCAN(min_cluster_size=3 if len(labels)>=12 else 2,
#                                   min_samples=1, metric="euclidean").fit_predict(X)
#   - Report n_clusters (excluding -1) and n_noise in the label string.
#   - Fallback if hdbscan not importable: return np.full(len(labels), -1), "HDBSCAN(unavailable)"
#
# ── C) summarize_clusters(assignments, cluster_col, top_terms_df) ─────────────
#   Returns: DataFrame with columns [cluster, n_subcategories, subcategories, representative_terms]
#   - Merge assignments with top_terms_df on "subcategory".
#   - For each cluster group: list subcategories and join up to 4 representative top_terms strings.
#
# Also implement (needed for cluster quality metrics):
# ── D) cluster_count(cluster_labels, include_noise=False) → int
# ── E) noise_count(cluster_labels) → int

def cluster_quality(X, cluster_labels, metric="cosine"):
    # TODO: filter out noise (-1 labels), compute silhouette_score; return nan on edge cases
    # my code
    # convert cluster_labels to numpy array, which is more efficient for filtering
    labels_array=np.array(cluster_labels)
    # filter out noise. the noise points are labeled as -1
    no_noise=labels_array!=-1
    # only keep the points that are not noise
    X_without_noise=X[no_noise]
    # only keep the labels that are not noise
    labels_without_noise=labels_array[no_noise]
    # get the number of clusters after filtering out noise
    number_of_clusters=cluster_count(labels_without_noise)
    # if there are less than 2 clusters, return nan
    if number_of_clusters<2:
        return np.nan
    # if the number of points is less than the number of clusters, return nan
    if len(labels_without_noise)<=number_of_clusters:
        return np.nan
    # compute the silhouette score
    try:
        score=silhouette_score(
            X_without_noise,
            labels_without_noise,
            metric=metric)
        return float(score)
    except:
        return np.nan
    

def cluster_count(cluster_labels, include_noise=False) -> int:
    # TODO: count distinct cluster labels (excluding -1 if include_noise=False)
    # my code
    # convert cluster_labels to numpy array, which is more efficient for filtering
    labels_array=np.array(cluster_labels)
    # get the unique labels
    unique_labels=set(labels_array.tolist())
    # decide whether to include noise
    if include_noise==False:
        # if exists noise, remove it
        if -1 in unique_labels:
            unique_labels.remove(-1)
    # return the number of unique labels
    return len(unique_labels)

def noise_count(cluster_labels) -> int:
    # TODO: count elements equal to -1
    # my code
    # convert cluster_labels to numpy array, which is more efficient for filtering
    labels_array=np.array(cluster_labels)
    # count the number of elements equal to -1
    number_of_noise_points=np.sum(labels_array==-1)
    # return the number of noise points
    return int(number_of_noise_points)

def summarize_clusters(assignments, cluster_col: str, top_terms_df) -> "pd.DataFrame":
    # TODO: merge, group by cluster_col, build summary rows
    # my code
    # merge the assignments with the top_terms_df
    merged_df=assignments.merge(top_terms_df,on="subcategory",how="left")
    # create a list to store the summary of clusters
    summary_row_list=[]
    # group by cluster_col
    for cluster_id, group_df in merged_df.groupby(cluster_col):
        # get all the subcategories in the cluster
        subcategories=group_df["subcategory"].tolist()
        # get the top terms from the cluster
        top_terms_list=(
            group_df["top_terms"]
            .dropna()
            .astype(str)
            .head(4)
            .tolist()
        )
        # create a summary row
        summary_row={
            "cluster":cluster_id,
            "n_subcategories":len(subcategories),
            "subcategories":", ".join(subcategories),
            "representative_terms":" | ".join(top_terms_list)
        }
        # add the summary row to the list
        summary_row_list.append(summary_row)
    # convert the list to a dataframe
    summary_df=pd.DataFrame(summary_row_list)
    # return the dataframe
    return summary_df

def run_hdbscan_diagnostic(X: np.ndarray, labels: list):
    try:
        import hdbscan as _hdbscan
        min_cluster_size = 3 if len(labels) >= 12 else 2
        # TODO: instantiate and fit HDBSCAN; build descriptive label string; return (out, label)
        # my code
        # instantiate HDBSCAN
        hdbscan_model=_hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=1,
            metric="euclidean"
        )
        # fit HDBSCAN
        out=hdbscan_model.fit_predict(X)
        # get the cluster labels
        method_label=(
            f"HDBSCAN(min_cluster_size={min_cluster_size}, "
            f"min_samples=1, metric=euclidean; "
            f"n_clusters={cluster_count(out)}, "
            f"n_noise={noise_count(out)})"
        )
        # return the cluster labels and the method label
        return out, method_label
    except ImportError:
        print("hdbscan not available; skipping density clustering diagnostic.")
        return np.full(len(labels), -1), "HDBSCAN(unavailable)"


### 6.2 Fit the clustering model

The cluster labels are a compact summary of the embedding map. Interpret them together with the heatmaps and the category support counts.


In [ ]:
# TODO (Category E — Guided sklearn): Run HDBSCAN clustering and build results tables.
# Estimated time: ~0.5 h
#
# Steps:
# 1. hdbscan_labels, hdbscan_method = run_hdbscan_diagnostic(X_emb, subcategories)
# 2. Build assignments DataFrame with columns:
#      subcategory, n_reviews, n_businesses, hdbscan_cluster
#    using data from subcategories list and corpus_df.
# 3. Compute sil_hdbscan = cluster_quality(X_emb, hdbscan_labels, metric="cosine")
# 4. Build cluster_metrics_df: one row per method with columns:
#      method, representation, n_clusters_excluding_noise, noise_points, silhouette
# 5. Print and display cluster_metrics_df.

# TODO: run HDBSCAN
hdbscan_labels, hdbscan_method=run_hdbscan_diagnostic(X_emb, subcategories)

# TODO: build assignments DataFrame
assignments=pd.DataFrame({
    "subcategory":subcategories,
    "hdbscan_cluster":hdbscan_labels
})
assignments=assignments.merge(
    corpus_df[["subcategory", "n_reviews", "n_businesses"]],
    on="subcategory",
    how="left"
)
assignments=assignments[
    ["subcategory", "n_reviews", "n_businesses", "hdbscan_cluster"]
]

# TODO: compute silhouette score
sil_hdbscan = cluster_quality(X_emb, hdbscan_labels, metric="cosine")

# TODO: build and display cluster_metrics_df
cluster_metrics_df = pd.DataFrame([
    {
        "method": hdbscan_method,
        "representation": embedding_method,
        "n_clusters_excluding_noise": cluster_count(hdbscan_labels),
        "noise_points": noise_count(hdbscan_labels),
        "silhouette": sil_hdbscan,
    }
])

print("HDBSCAN method:", hdbscan_method)
print(
    f"✓ HDBSCAN: {cluster_count(hdbscan_labels)} clusters, "
    f"{noise_count(hdbscan_labels)} noise points"
)

display(assignments)
display(cluster_metrics_df)  
    

# --- Validation ---
assert hdbscan_labels is not None
assert isinstance(assignments, pd.DataFrame) and "subcategory" in assignments.columns
print(f"\u2713 HDBSCAN: {cluster_count(hdbscan_labels)} clusters, {noise_count(hdbscan_labels)} noise points")


### 6.3 Plot cluster assignments on the same 2-D layout

Using the same layout for several cluster labels makes method differences easier to inspect. You should avoid treating color groups as self-evident; each cluster needs a short evidence-based interpretation.

In [ ]:
plot_2d_map(
    coords_emb, subcategories,
    f"2-D layout with HDBSCAN cluster labels — {hdbscan_method}",
    "layout_with_hdbscan_clusters.png",
    cluster_labels=hdbscan_labels,
)

### 6.4 Cluster summary tables

Cluster summaries translate labels into readable evidence. The representative terms are intentionally compact; use them together with the similarity heatmaps.


In [ ]:
hdbscan_summary = summarize_clusters(
    assignments[["subcategory", "hdbscan_cluster"]],
    "hdbscan_cluster",
    top_terms_df,
)
hdbscan_summary.to_csv(OUTPUT_DIR / "cluster_summary_hdbscan.csv", index=False)
print("HDBSCAN cluster summary:")
display(hdbscan_summary)

## Downstream handoff: cuisine/subcategory map artifacts

Task 2 produces reusable cuisine/subcategory similarity, cluster, and embedding artifacts. Tasks 3-6 can use these files to choose coherent categories for dish discovery, carry cuisine-neighborhood context into evidence search, summarize temporal patterns by category cluster, or build category-map features for later prediction tasks.

### 7.1 Save standardized similarity artifacts

These files are the cleanest machine-readable outputs from Task 2: the TF-IDF map, the embedding map, and one cluster assignment table.


In [ ]:
# TODO (Category E — Guided): Save the similarity matrices as labeled DataFrames.
# Estimated time: ~0.5 h
#
# Expected output:
#   ARTIFACT_DIR / "subcategory_similarity_tfidf.csv"      (index=subcategories, columns=subcategories)
#   ARTIFACT_DIR / "subcategory_similarity_embedding.csv"  (same structure)
#
# Steps:
# 1. subcategory_similarity_tfidf = pd.DataFrame(S_tfidf, index=subcategories, columns=subcategories)
# 2. subcategory_similarity_embedding = pd.DataFrame(S_emb, index=subcategories, columns=subcategories)
# 3. Save both CSVs to OUTPUT_DIR.

# TODO: build and save both similarity DataFrames
subcategory_similarity_tfidf = pd.DataFrame(
    S_tfidf,
    index=subcategories,
    columns=subcategories
)
subcategory_similarity_embedding  = pd.DataFrame(
    S_emb,
    index=subcategories,
    columns=subcategories
)
# 把 TF-IDF 相似度 DataFrame 保存成 CSV 文件
subcategory_similarity_tfidf.to_csv(
    OUTPUT_DIR / "subcategory_similarity_tfidf.csv"
)


# 把 embedding 相似度 DataFrame 保存成 CSV 文件
subcategory_similarity_embedding.to_csv(
    OUTPUT_DIR / "subcategory_similarity_embedding.csv"
)


# --- Validation ---
assert (OUTPUT_DIR / "subcategory_similarity_tfidf.csv").exists()
assert (OUTPUT_DIR / "subcategory_similarity_embedding.csv").exists()
print("\u2713 Similarity matrices saved")


### 7.2 Save cluster and cuisine-map summary artifacts

The summary table collects category support, top terms, cluster labels, and representation metadata in one place.


In [ ]:
# TODO (Category D — Medium pipeline): Build the downstream handoff artifact tables.
# Estimated time: ~0.5 h
#
# Expected output files:
#   OUTPUT_DIR / "subcategory_clusters.csv"     — assignments merged with top_terms_df
#   OUTPUT_DIR / "cuisine_map_summary.csv"      — corpus_df metadata merged with cluster info
#
# Steps:
# 1. subcategory_clusters = assignments.copy()
#    If top_terms_df is available, merge it on "subcategory" (left join).
#    Save to OUTPUT_DIR / "subcategory_clusters.csv".
#
# 2. cuisine_map_summary = corpus_df[["subcategory","n_reviews","n_businesses","avg_review_stars"]].copy()
#    Merge with top_terms_df if available; merge with assignments cluster columns if available.
#    Sort by n_reviews descending.
#    Save to OUTPUT_DIR / "cuisine_map_summary.csv".
#
# 3. Print "Saved: ..." for each file and display head(10).

# TODO: build subcategory_clusters and save
subcategory_clusters = assignments.copy()
# 判断 top_terms_df 是否已经存在
if "top_terms_df" in globals():

    # 把每个 subcategory 的 top TF-IDF terms 合并进 cluster 表
    subcategory_clusters = subcategory_clusters.merge(
        top_terms_df,
        on="subcategory",
        how="left"
    )

# 设置 subcategory_clusters 的保存路径
subcategory_clusters_path = OUTPUT_DIR / "subcategory_clusters.csv"
# 把 subcategory_clusters 保存成 CSV 文件
subcategory_clusters.to_csv(subcategory_clusters_path, index=False)
# 打印保存路径
print("Saved:", subcategory_clusters_path)
# 显示 subcategory_clusters 的前 10 行
display(subcategory_clusters.head(10))

# TODO: build cuisine_map_summary and save
cuisine_map_summary= corpus_df[
    ["subcategory", "n_reviews", "n_businesses", "avg_review_stars"]
].copy()

# 判断 top_terms_df 是否已经存在
if "top_terms_df" in globals():

    # 把 top terms 合并进 cuisine summary 表
    cuisine_map_summary = cuisine_map_summary.merge(
        top_terms_df,
        on="subcategory",
        how="left"
    )


# 判断 assignments 是否已经存在
if "assignments" in globals():

    # 从 assignments 中找出 subcategory 和所有 cluster 相关列
    cluster_columns = [
        col for col in assignments.columns
        if col == "subcategory" or "cluster" in col
    ]

    # 把 cluster 信息合并进 cuisine summary 表
    cuisine_map_summary = cuisine_map_summary.merge(
        assignments[cluster_columns],
        on="subcategory",
        how="left"
    )

# 按照 n_reviews 从多到少排序
cuisine_map_summary = cuisine_map_summary.sort_values(
    "n_reviews",
    ascending=False
)

# 设置 cuisine_map_summary 的保存路径
cuisine_map_summary_path = OUTPUT_DIR / "cuisine_map_summary.csv"

# 把 cuisine_map_summary 保存成 CSV 文件
cuisine_map_summary.to_csv(cuisine_map_summary_path, index=False)

# 打印保存路径
print("Saved:", cuisine_map_summary_path)

# 显示 cuisine_map_summary 的前 10 行
display(cuisine_map_summary.head(10))
# --- Validation ---
assert (OUTPUT_DIR / "subcategory_clusters.csv").exists()
assert (OUTPUT_DIR / "cuisine_map_summary.csv").exists()
print("\u2713 Handoff artifacts saved")


### Handoff checkpoint

Before moving to later tasks, verify that your saved Task 2 artifacts describe both the similarity structure and the evidence behind it. Task 3 can use these outputs to choose a coherent category neighborhood for dish phrase discovery, and Task 4 can use the compact cuisine/category map to justify which dish evidence should be compared.

## Visual Evidence and Interpretation

Visualizations in this notebook are evidence, not decoration. Each figure should support interpretation, method comparison, diagnostic analysis, or communication of a discovered Yelp restaurant-intelligence pattern.

In a student report, explain what each selected figure reveals and one limitation of the visualization, especially when smoke-test data are small or sparse.

### Visual evidence helpers

The helper functions below keep plots readable by ordering labels, truncating long names, and capping graph size. This matters because unreadable visualizations do not count as useful evidence.

In [ ]:
# Visual evidence layer: ordered similarity heatmaps and method comparison.
import textwrap
VIS_MAX_LABELS = 35
VIS_GRAPH_MAX_NODES = 25
VIS_GRAPH_TOP_K = 2

def _short_label(value, width=18):
    return "\n".join(textwrap.wrap(str(value), width=width)) or str(value)

def _cluster_order_from_similarity_df(sim_df: pd.DataFrame) -> list[str]:
    labels = sim_df.index.tolist()
    if len(labels) <= 2:
        return labels
    try:
        from scipy.spatial.distance import squareform
        from scipy.cluster.hierarchy import linkage, leaves_list
        S = sim_df.to_numpy(dtype=float)
        D = np.clip(1 - S, 0, 2)
        np.fill_diagonal(D, 0)
        order = leaves_list(linkage(squareform(D, checks=False), method="average"))
        return [labels[i] for i in order]
    except Exception as exc:
        print(f"Could not compute hierarchical order; using mean-similarity fallback: {exc}")
        means = sim_df.mask(np.eye(len(sim_df), dtype=bool)).mean(axis=1).sort_values(ascending=False)
        return means.index.tolist()

def _plot_ordered_similarity(sim_df: pd.DataFrame, title: str, filename: str, order: list[str] | None = None):
    if sim_df is None or sim_df.empty or sim_df.shape[0] < 2:
        print(f"Skipping {filename}: not enough categories for a similarity heatmap.")
        return False
    if order is None:
        order = _cluster_order_from_similarity_df(sim_df)
    order = [x for x in order if x in sim_df.index and x in sim_df.columns][:VIS_MAX_LABELS]
    if len(order) < 2:
        print(f"Skipping {filename}: not enough ordered labels.")
        return False
    plot_df = sim_df.loc[order, order]
    fig_size = max(7, min(15, len(order) * 0.45 + 3))
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    vmin = -1 if np.nanmin(plot_df.to_numpy()) < 0 else 0
    im = ax.imshow(plot_df.to_numpy(), vmin=vmin, vmax=1, aspect="auto")
    ax.set_xticks(np.arange(len(order)))
    ax.set_yticks(np.arange(len(order)))
    ax.set_xticklabels([_short_label(x, 15) for x in order], rotation=90, fontsize=7)
    ax.set_yticklabels([_short_label(x, 16) for x in order], fontsize=7)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="cosine similarity", fraction=0.046, pad=0.04)
    fig.tight_layout()
    out_path = OUTPUT_DIR / filename
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved:", out_path)
    return True

### Ordered heatmaps and method-comparison table

These figures and tables make representation comparison explicit. Use the same TF-IDF order where appropriate so visual differences reflect representation differences rather than arbitrary label ordering.

In [ ]:
# TODO (Category D — Medium pipeline): Plot ordered similarity heatmaps and build comparison table.
# Estimated time: ~1 h
#
# Expected inputs:
#   S_tfidf, S_emb     : similarity matrices (numpy arrays)
#   subcategories      : list of subcategory name strings
#   OUTPUT_DIR         : Path for saving artifacts
#
# Steps:
# 1. Build similarity_frames = {"tfidf": pd.DataFrame(S_tfidf,...), "embedding": pd.DataFrame(S_emb,...)}
# 2. For each matrix, compute a hierarchical cluster order (use hierarchical_order_from_similarity
#    or scipy linkage directly).
# 3. Plot an ordered heatmap for TF-IDF similarity, saved to "similarity_tfidf_clustered_heatmap.png".
# 4. Plot an ordered heatmap for embedding similarity, saved to "similarity_embedding_clustered_heatmap.png".
# 5. Build a comparison table showing for each method: n_subcategories, top-5 most-similar pairs,
#    min/mean/max similarity, and save to OUTPUT_DIR / "similarity_method_comparison.csv".
#    Hint: for top-5 similar pairs, mask the diagonal (set to -1) and use np.unravel_index(argsort).

# TODO: build similarity_frames dict
similarity_frames = {
    "tfidf": pd.DataFrame(S_tfidf, index=subcategories, columns=subcategories),
    "embedding": pd.DataFrame(S_emb, index=subcategories, columns=subcategories),
}

# TODO: compute cluster orders and plot ordered heatmaps for both methods
# 计算 TF-IDF 相似度矩阵的层次聚类排序
order_tfidf_clustered = hierarchical_order_from_similarity(S_tfidf)


# 计算 embedding 相似度矩阵的层次聚类排序
order_embedding_clustered = hierarchical_order_from_similarity(S_emb)

# 画 TF-IDF 的 ordered heatmap，并保存图片
plot_similarity_heatmap(
    S_tfidf,
    subcategories,
    "Ordered TF-IDF similarity heatmap",
    "similarity_tfidf_clustered_heatmap.png",
    order=order_tfidf_clustered
)

# 画 embedding 的 ordered heatmap，并保存图片
plot_similarity_heatmap(
    S_emb,
    subcategories,
    "Ordered embedding similarity heatmap",
    "similarity_embedding_clustered_heatmap.png",
    order=order_embedding_clustered
)

# 定义一个辅助函数，用来找出相似度最高的前几个 subcategory pairs
def get_top_similar_pairs(S, labels, top_n=5):

    # 把相似度矩阵复制一份，避免修改原始矩阵
    S_copy = S.copy()

    # 创建上三角 mask，只保留不重复的 pair，不包括对角线
    upper_triangle_mask = np.triu(np.ones(S_copy.shape, dtype=bool), k=1)

    # 取出上三角中的相似度数值
    pair_scores = S_copy[upper_triangle_mask]

    # 找出上三角中每个 pair 对应的行列索引
    pair_indices = np.argwhere(upper_triangle_mask)

    # 按相似度从高到低排序，取前 top_n 个
    top_order = np.argsort(pair_scores)[::-1][:top_n]

    # 创建一个空列表，用来存放 top pairs 的字符串
    top_pairs = []

    # 遍历 top pair 的排序结果
    for pair_order_index in top_order:

        # 取出这个 pair 的第一个 subcategory 的位置
        i = pair_indices[pair_order_index][0]

        # 取出这个 pair 的第二个 subcategory 的位置
        j = pair_indices[pair_order_index][1]

        # 取出这个 pair 的相似度分数
        score = pair_scores[pair_order_index]

        # 把 pair 和分数拼成可读字符串
        pair_text = f"{labels[i]} ↔ {labels[j]} ({score:.3f})"

        # 把这个 pair 加入列表
        top_pairs.append(pair_text)

    # 返回 top pairs，用分号连接成一个字符串
    return "; ".join(top_pairs)


# 创建一个空列表，用来存放每种方法的比较结果
comparison_rows = []


# 遍历每一种相似度方法
for method_name, similarity_df in similarity_frames.items():

    # 把当前方法的相似度 DataFrame 转成 numpy array
    S_current = similarity_df.values

    # 创建上三角 mask，只统计不同 subcategory 之间的相似度
    upper_triangle_mask = np.triu(np.ones(S_current.shape, dtype=bool), k=1)

    # 取出非对角线、非重复 pair 的相似度
    off_diagonal_scores = S_current[upper_triangle_mask]

    # 计算 top-5 最相似的 subcategory pairs
    top_5_pairs = get_top_similar_pairs(S_current, subcategories, top_n=5)

    # 创建当前方法的一行比较结果
    comparison_row = {
        "method": method_name,
        "n_subcategories": len(subcategories),
        "top_5_most_similar_pairs": top_5_pairs,
        "min_similarity": float(off_diagonal_scores.min()),
        "mean_similarity": float(off_diagonal_scores.mean()),
        "max_similarity": float(off_diagonal_scores.max()),
    }

    # 把当前方法的结果加入列表
    comparison_rows.append(comparison_row)



# TODO: build and save method comparison table
method_comparison = pd.DataFrame(comparison_rows)

# 设置比较表的保存路径
method_comparison_path = OUTPUT_DIR / "similarity_method_comparison.csv"

# 把比较表保存成 CSV 文件
method_comparison.to_csv(method_comparison_path, index=False)

# 显示比较表
display(method_comparison)

# --- Validation ---
assert (OUTPUT_DIR / "similarity_tfidf_clustered_heatmap.png").exists()
assert (OUTPUT_DIR / "similarity_embedding_clustered_heatmap.png").exists()
assert (OUTPUT_DIR / "similarity_method_comparison.csv").exists()
print("\u2713 Ordered heatmaps and comparison table saved")


## Required comparison and interpretation

Your report should answer the following using concrete examples from your tables and figures:

1. Which representation produced the most interpretable cuisine/category neighbourhoods?
2. Where did TF-IDF and embeddings agree? Use similarity heatmap or HDBSCAN cluster examples.
3. Where did they disagree, and what might explain the disagreement?
4. What does HDBSCAN find? Are the clusters interpretable? If HDBSCAN labels many points
   as noise (−1), what does that suggest about the representation or the data?
5. How could these results inform Task 3 dish discovery or Task 4 restaurant ranking?

In [ ]:
# TODO (Category D — Medium pipeline): Assemble the cross-method comparison summary.
# Estimated time: ~0.5 h
#
# Build a comparison_summary dict and save it as a CSV.
#
# Required keys in comparison_summary:
#   category_focus_mode, max_final_reviews_per_business, requires_sentence_transformer_real_mode,
#   embedding_used_fallback, n_subcategories, n_reviews_subset, n_review_subcategory_pairs,
#   modern_representation_method, tfidf_top_pair (the most similar subcategory pair by TF-IDF),
#   embedding_top_pair (the most similar pair by embedding similarity),
#   any other fields you consider informative for comparing the two pipelines
#
# Save comparison_summary as OUTPUT_DIR / "task2_comparison_summary.csv".
# Display it (pd.DataFrame([comparison_summary])).
#
# Hint for top pair: mask S_tfidf diagonal and use np.unravel_index(S_tfidf_masked.argmax(), shape).


# 定义一个辅助函数，用来找出相似度矩阵中最相似的一对 subcategory
def get_top_pair_from_similarity(S, labels):

    # 复制相似度矩阵，避免修改原始矩阵
    S_masked = S.copy()

    # 把对角线设置为 -1，避免自己和自己成为最高相似 pair
    np.fill_diagonal(S_masked, -1)

    # 找到矩阵中最大值的位置
    top_pair_index = np.unravel_index(S_masked.argmax(), S_masked.shape)

    # 取出第一个 subcategory 的 index
    i = top_pair_index[0]

    # 取出第二个 subcategory 的 index
    j = top_pair_index[1]

    # 取出这一对 subcategory 的相似度
    score = S_masked[i, j]

    # 把 pair 和分数拼成字符串返回
    return f"{labels[i]} ↔ {labels[j]} ({score:.3f})"


# 计算 TF-IDF 相似度最高的一对 subcategory
tfidf_top_pair = get_top_pair_from_similarity(S_tfidf, subcategories)


# 计算 embedding 相似度最高的一对 subcategory
embedding_top_pair = get_top_pair_from_similarity(S_emb, subcategories)


# 判断 embedding 方法是否使用了 fallback
embedding_used_fallback = "fallback" in embedding_method.lower()


# 计算 review-subcategory pair 的数量
n_review_subcategory_pairs = len(pairs_top)


# 计算 pairs_top 中不重复 review 的数量
n_unique_reviews_in_pairs = pairs_top["review_id"].nunique()


# 计算平均每条 review 有多少个 subcategory label
mean_labels_per_review = pairs_top.groupby("review_id")["subcategory"].nunique().mean()

# TODO: build comparison_summary dict with all required keys
comparison_summary = {

    # 记录当前使用的是 broad 还是 cuisine-focused 类别模式
    "category_focus_mode": CATEGORY_FOCUS_MODE,

    # 记录每个 business 最多保留多少条最终 review
    "max_final_reviews_per_business": MAX_FINAL_REVIEWS_PER_BUSINESS,

    # 记录真实模式下是否要求使用 sentence-transformer
    "requires_sentence_transformer_real_mode": REQUIRE_SENTENCE_TRANSFORMER,

    # 记录 embedding 是否使用了 fallback 方法
    "embedding_used_fallback": embedding_used_fallback,

    # 记录最终 corpus 中 subcategory 的数量
    "n_subcategories": len(corpus_df),

    # 记录最终 review subset 的数量
    "n_reviews_subset": len(reviews_df),

    # 记录 review-subcategory pair 的数量
    "n_review_subcategory_pairs": n_review_subcategory_pairs,

    # 记录 pairs_top 中不重复 review 的数量
    "n_unique_reviews_in_pairs": n_unique_reviews_in_pairs,

    # 记录平均每条 review 对应多少个 subcategory label
    "mean_labels_per_review": mean_labels_per_review,

    # 记录现代 embedding pipeline 使用的方法
    "modern_representation_method": embedding_method,

    # 记录 TF-IDF pipeline 中最相似的一对 subcategory
    "tfidf_top_pair": tfidf_top_pair,

    # 记录 embedding pipeline 中最相似的一对 subcategory
    "embedding_top_pair": embedding_top_pair,

    # 记录 TF-IDF 相似度矩阵的平均相似度
    "tfidf_mean_similarity": float(S_tfidf[np.triu_indices_from(S_tfidf, k=1)].mean()),

    # 记录 embedding 相似度矩阵的平均相似度
    "embedding_mean_similarity": float(S_emb[np.triu_indices_from(S_emb, k=1)].mean()),

    # 记录 HDBSCAN cluster 数量，如果 hdbscan_labels 已经存在
    "hdbscan_n_clusters": cluster_count(hdbscan_labels) if "hdbscan_labels" in globals() else np.nan,

    # 记录 HDBSCAN noise 数量，如果 hdbscan_labels 已经存在
    "hdbscan_noise_points": noise_count(hdbscan_labels) if "hdbscan_labels" in globals() else np.nan,

    # 记录 HDBSCAN silhouette，如果 sil_hdbscan 已经存在
    "hdbscan_silhouette": sil_hdbscan if "sil_hdbscan" in globals() else np.nan,
}

# TODO: save and display
# pd.DataFrame([comparison_summary]).to_csv(OUTPUT_DIR / "task2_comparison_summary.csv", index=False)
# 把 comparison_summary 转成单行 DataFrame
comparison_summary_df = pd.DataFrame([comparison_summary])


# 设置保存路径
comparison_summary_path = OUTPUT_DIR / "task2_comparison_summary.csv"


# 保存 comparison summary CSV
comparison_summary_df.to_csv(comparison_summary_path, index=False)


# 显示 comparison summary
display(comparison_summary_df)


# --- Validation ---
assert comparison_summary.get("n_subcategories") == len(corpus_df)
assert (OUTPUT_DIR / "task2_comparison_summary.csv").exists()
print("\u2713 Comparison summary saved with", len(comparison_summary), "metrics")


**Insights! 💡**

To extend the mental model of text similarity and clustering techniques across different "eras," here is a broader historical perspective on how the field moved from explicit rules to learned geometry:

**The "Pre-History" (1950s – 1970s):** Rule-Based & Symbolic AI
- **The Tech:** Regular Expressions, Context-Free Grammars, and manually curated ontologies (like early WordNet).
- **The Vibe:** Everything was a strict "if-then" rule. Similarity was based on shared taxonomy nodes, not learned from data. Two items were "similar" only if an expert explicitly said so.

**Between TF-IDF and LDA (Early 2000s):** The Latent Semantic Era
- **The Tech:** LSA/LSI (Latent Semantic Indexing).
- **The Gap:** TF-IDF was just counting; LSA was the bridge—it used Singular Value Decomposition (SVD) to discover that "doctor" and "physician" often appear in the same documents, even if they aren't the same word. It was the first real "compression" of meaning, and it underpins the TruncatedSVD fallback path in this very notebook.

**Between LSA and Embeddings (2010 – 2015):** The Neural Transition
- **The Tech:** RNNs, LSTMs, and Word2Vec.
- **The Gap:** LSA found "similar documents," but Word2Vec (2013) found "similar words" in a continuous vector space. The limitation: it represented individual words, not full sentences or documents. TF-IDF similarity of long category documents was still often competitive.

**The "Next Era" (2024 – Future):** Agentic Reasoning & World Models
- **The Tech:** LLM agents, multimodal models, and long-context embedding/retrieval systems.
- **The Vibe:** We are moving past static embeddings (vectors) toward reasoning chains. Instead of just finding a "similar" vector, the model interprets ***intent***, executes tools, and maintains a "state" over time.

**The Updated Timeline:**
- Symbolic Era (Logic/Rules)
- Statistical Era (TF-IDF/N-grams)
- Latent Era (LSA/LSI)
- Probabilistic Era (LDA/Bayesian)
- Sequential Era (RNNs/LSTMs)
- Embedding/Transformer Era (Word2Vec → BERT → Sentence-BERT)
- Agentic Era (Reasoning/Autonomy)


## 8. Required report deliverables

Your PDF report should be approximately **2–3 pages** and include:

- **TF-IDF similarity**: which categories are lexically closest?
- **Embedding similarity**: how does the semantic map compare?
- **Agreement case**: one pair where both methods agree and why.
- **Disagreement case**: one pair where methods disagree; explain the likely cause.
- **Clustering result**: what cluster structure emerges, and is it interpretable?
- **Representation choice**: which method would you trust most for a cuisine map, and why?


## Final artifact validation

Run this cell before submission. It checks that required artifacts exist, are non-empty, and satisfy the most important schema expectations. Optional artifacts are reported but do not block submission.

In [ ]:
# Final artifact and schema validation for Task 2.
expected_artifacts = [
    "subcategory_similarity_tfidf.csv",
    "subcategory_similarity_embedding.csv",
    "subcategory_clusters.csv",
    "cuisine_map_summary.csv",
]
optional_csv_artifacts = [
    "task2_cluster_assignments.csv",
    "task2_cluster_metrics.csv",
    "task2_sampling_diagnostics.csv",
    "task2_sampling_top_businesses.csv",
    "task2_top_tfidf_terms_by_subcategory.csv",
]
optional_json_artifacts = [
    "task2_comparison_summary.json",
]
optional_table_bases = [
    "task2_reviews_subset",
    "task2_review_subcategory_pairs",
    "task2_subcategory_corpus",
]

missing = [f for f in expected_artifacts if not (OUTPUT_DIR / f).exists()]
if missing:
    print("WARNING: Missing required artifacts:", missing)
else:
    print("OK: All required artifacts present.")

def report_csv(path: Path):
    try:
        nrows = len(pd.read_csv(path))
        print(f"  {path.name}: {nrows} rows")
    except Exception:
        print(f"  {path.name}: present, row count unavailable")

print("Current Task 2 artifact check:")
for name in expected_artifacts + optional_csv_artifacts:
    p = OUTPUT_DIR / name
    if p.exists():
        report_csv(p)

for name in optional_json_artifacts:
    p = OUTPUT_DIR / name
    if p.exists():
        print(f"  {p.name}: present")

for base in optional_table_bases:
    parquet_path = OUTPUT_DIR / f"{base}.parquet"
    csv_path = OUTPUT_DIR / f"{base}.csv"
    if parquet_path.exists():
        print(f"  {parquet_path.name}: present")
    elif csv_path.exists():
        report_csv(csv_path)

## Common pitfalls to avoid

Watch for these issues as you prepare your report:

- Treating the task as only a clustering exercise instead of a representation comparison.
- Selecting categories that are too generic to produce an informative map.
- Ignoring sampling diagnostics when one or two businesses dominate the subset.
- Claiming embeddings are better without concrete evidence from heatmaps, clusters, or examples.
- Hiding weak clustering results instead of explaining what they reveal about the representation and sample size.
- Omitting important parameters such as category focus, sampling caps, vectorizer settings, model name, and fallback status.


## xiaofei
## 7.1 DishRAG Extension — Category-Level Retrieval Layer

This extension reframes Task 2 as a category-level retrieval layer for DishRAG. 
Instead of optimizing for prettier clusters, we build robust cuisine-neighborhood evidence 
for downstream evidence-grounded recommendation.

In [ ]:
# DishRAG category retrieval extension configuration

from collections import Counter, defaultdict
import re
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.decomposition import TruncatedSVD

DISHRAG_DIR = OUTPUT_DIR / "dishrag_category_retrieval"
DISHRAG_DIR.mkdir(parents=True, exist_ok=True)

TOP_K_NEIGHBORS = 5
HYBRID_EMBEDDING_WEIGHT = 0.60
HYBRID_TFIDF_WEIGHT = 0.40

PRIMARY_MIN_REVIEWS = 100
N_BOOTSTRAPS = 20
BOOTSTRAP_TOP_K = 5
STABILITY_THRESHOLD = 0.80

print("DishRAG extension output directory:", DISHRAG_DIR)

In [ ]:
# Named-entity / brand-name filtering for cleaner lexical evidence

GENERIC_BUSINESS_WORDS = {
    "restaurant", "restaurants", "food", "foods", "bar", "bars",
    "cafe", "coffee", "tea", "grill", "grille", "kitchen",
    "bistro", "house", "place", "shop", "market", "diner",
    "the", "and", "inc", "llc", "co", "company"
}

MANUAL_ENTITY_STOPWORDS = {
    "benihana", "whataburger", "fatburger", "outback",
    "changs", "pf", "pappadeaux", "taiko", "cherryblossom",
    "payard", "savoy", "ticoz", "bomberos", "rumbi",
    "emeril", "commander", "lucille", "yupha", "chevy",
    "famous", "dave", "mortons", "morton"
}

PROTECTED_CUISINE_TERMS = {
    "pizza", "mexican", "sushi", "chinese", "italian", "japanese",
    "thai", "greek", "mediterranean", "indian", "korean",
    "vietnamese", "barbeque", "seafood", "burger", "burgers",
    "french", "tex", "mex", "latin", "american", "middle",
    "eastern", "vegetarian", "asian", "fusion", "cajun",
    "creole", "caribbean", "southern", "steakhouse", "steakhouses"
}

ENTITY_MIN_BUSINESS_NAME_FREQ = 5

PROTECTED_DISH_EVIDENCE_TERMS = {
    "asada", "carne", "taco", "tacos", "burrito", "burritos",
    "salsa", "guacamole", "fajita", "fajitas", "enchilada", "enchiladas",
    "bbq", "barbecue", "brisket", "ribs", "pulled", "pork",
    "gyro", "gyros", "tzatziki", "falafel", "hummus", "shawarma",
    "naan", "masala", "curry", "paneer", "tandoori",
    "pho", "banh", "ramen", "sushi", "nigiri", "miso",
    "basil", "noodle", "noodles", "dumpling", "dumplings",
    "pizza", "pasta", "alfredo", "risotto",
    "crawfish", "oyster", "shrimp", "seafood"
}

def tokenize_name_for_entities(name: str) -> list[str]:
    text = str(name).lower()
    tokens = re.findall(r"[a-z][a-z']+", text)
    tokens = [
        token.strip("'")
        for token in tokens
        if len(token.strip("'")) >= 3
    ]
    return tokens

def build_entity_stopwords(
    pairs_df: pd.DataFrame,
    min_business_name_freq: int = ENTITY_MIN_BUSINESS_NAME_FREQ
) -> set[str]:
    name_counter = Counter()

    if "business_name" in pairs_df.columns:
        business_names = pairs_df["business_name"].dropna().astype(str).unique()
        for name in business_names:
            for token in tokenize_name_for_entities(name):
                if token not in GENERIC_BUSINESS_WORDS:
                    name_counter[token] += 1

    frequent_name_tokens = {
        token
        for token, count in name_counter.items()
        if count >= min_business_name_freq
    }

    entity_stopwords = set(MANUAL_ENTITY_STOPWORDS) | frequent_name_tokens
    entity_stopwords = {
        token
        for token in entity_stopwords
        if token not in GENERIC_BUSINESS_WORDS
        and token not in PROTECTED_CUISINE_TERMS
        and token not in PROTECTED_DISH_EVIDENCE_TERMS
    }

    return entity_stopwords

entity_stopwords = build_entity_stopwords(pairs_top)

print("Number of entity stopwords:", len(entity_stopwords))
print("Sample entity stopwords:", sorted(list(entity_stopwords))[:50])

In [ ]:
# Recompute TF-IDF after named-entity filtering

vec_tfidf_filtered, X_tfidf_filtered = vectorize_corpus(
    corpus_df,
    use_idf=True,
    extra_stop_words=entity_stopwords
)

S_tfidf_filtered = cosine_similarity(X_tfidf_filtered)
order_tfidf_filtered = hierarchical_order_from_similarity(S_tfidf_filtered)

filtered_tfidf_path = DISHRAG_DIR / "similarity_tfidf_entity_filtered_matrix.csv"
pd.DataFrame(
    S_tfidf_filtered,
    index=subcategories,
    columns=subcategories
).to_csv(filtered_tfidf_path)

top_terms_filtered_df = top_terms_for_rows(
    X_tfidf_filtered,
    vec_tfidf_filtered,
    subcategories,
    n=TOP_TERMS_PER_SUBCATEGORY
)

top_terms_filtered_df.to_csv(
    DISHRAG_DIR / "top_tfidf_terms_entity_filtered_by_subcategory.csv",
    index=False
)

print("Saved:", filtered_tfidf_path)
display(top_terms_filtered_df.head(25))

In [ ]:
# Hybrid cosine similarity for DishRAG category-neighborhood retrieval

from scipy.sparse import csr_matrix, hstack

def build_hybrid_cosine_matrix(
    X_embedding,
    X_tfidf,
    embedding_weight: float,
    tfidf_weight: float,
):
    if embedding_weight < 0 or tfidf_weight < 0:
        raise ValueError("Hybrid weights must be non-negative.")
    if embedding_weight + tfidf_weight <= 0:
        raise ValueError("At least one hybrid weight must be positive.")

    X_embedding_norm = normalize(X_embedding)
    X_tfidf_norm = normalize(X_tfidf)

    X_hybrid_features = hstack(
        [
            np.sqrt(embedding_weight) * csr_matrix(X_embedding_norm),
            np.sqrt(tfidf_weight) * X_tfidf_norm,
        ],
        format="csr",
    )
    X_hybrid_features = normalize(X_hybrid_features)

    S_hybrid_cosine = cosine_similarity(X_hybrid_features)
    np.fill_diagonal(S_hybrid_cosine, 1.0)
    return X_hybrid_features, S_hybrid_cosine

X_hybrid_features, S_hybrid = build_hybrid_cosine_matrix(
    X_emb,
    X_tfidf_filtered,
    HYBRID_EMBEDDING_WEIGHT,
    HYBRID_TFIDF_WEIGHT,
)

hybrid_matrix_path = DISHRAG_DIR / "hybrid_category_similarity_matrix.csv"
hybrid_cosine_matrix_path = DISHRAG_DIR / "hybrid_category_cosine_similarity_matrix.csv"
hybrid_matrix_df = pd.DataFrame(
    S_hybrid,
    index=subcategories,
    columns=subcategories,
)
hybrid_matrix_df.to_csv(hybrid_matrix_path)
hybrid_matrix_df.to_csv(hybrid_cosine_matrix_path)

print("Saved:", hybrid_matrix_path)
print("Saved:", hybrid_cosine_matrix_path)
print("Hybrid cosine matrix shape:", S_hybrid.shape)

In [ ]:
# Build DishRAG top-k category neighbor table

def rank_of_neighbor(S: np.ndarray, i: int, j: int) -> int:
    order = [idx for idx in np.argsort(-S[i]) if idx != i]
    return order.index(j) + 1

def build_hybrid_neighbor_table(
    S_hybrid,
    S_tfidf_filtered,
    S_emb,
    labels,
    top_terms_filtered_df,
    top_k=5
):
    top_terms_map = dict(
        zip(
            top_terms_filtered_df["subcategory"],
            top_terms_filtered_df["top_terms"]
        )
    )

    rows = []

    for i, source in enumerate(labels):
        neighbor_indices = [
            j for j in np.argsort(-S_hybrid[i])
            if j != i
        ][:top_k]

        for rank, j in enumerate(neighbor_indices, start=1):
            neighbor = labels[j]

            rows.append({
                "source_category": source,
                "neighbor_category": neighbor,
                "hybrid_rank": rank,
                "hybrid_cosine_score": float(S_hybrid[i, j]),
                "embedding_score": float(S_emb[i, j]),
                "filtered_tfidf_score": float(S_tfidf_filtered[i, j]),
                "embedding_rank": rank_of_neighbor(S_emb, i, j),
                "filtered_tfidf_rank": rank_of_neighbor(S_tfidf_filtered, i, j),
                "source_filtered_terms": top_terms_map.get(source, ""),
                "neighbor_filtered_terms": top_terms_map.get(neighbor, ""),
            })

    return pd.DataFrame(rows)

hybrid_neighbors_df = build_hybrid_neighbor_table(
    S_hybrid,
    S_tfidf_filtered,
    S_emb,
    subcategories,
    top_terms_filtered_df,
    top_k=TOP_K_NEIGHBORS
)

hybrid_neighbors_path = DISHRAG_DIR / "hybrid_category_neighbors_topk.csv"
hybrid_neighbors_df.to_csv(hybrid_neighbors_path, index=False)

print("Saved:", hybrid_neighbors_path)
display(hybrid_neighbors_df.head(30))

In [ ]:
# Multi-label vs primary-label sensitivity analysis

def normalize_categories_flexible(cats):
    if isinstance(cats, (list, tuple, set, np.ndarray, pd.Series)):
        return [str(c).strip() for c in list(cats) if str(c).strip()]

    if cats is None or (isinstance(cats, float) and pd.isna(cats)):
        return []

    text = str(cats).strip()

    if text.startswith("[") and text.endswith("]"):
        matches = re.findall(r"'([^']+)'|\"([^\"]+)\"", text)
        parsed = [
            single_quoted or double_quoted
            for single_quoted, double_quoted in matches
            if (single_quoted or double_quoted).strip()
        ]
        if parsed:
            return [category.strip() for category in parsed]

    return [
        part.strip().strip("'\"")
        for part in re.split(r",|;", text)
        if part.strip().strip("'\"")
    ]

def primary_subcategory_from_raw_categories(raw_categories, selected_categories):
    cats = normalize_categories_flexible(raw_categories)
    selected_set = set(selected_categories)

    for cat in cats:
        if cat in selected_set:
            return cat

    return None

def build_primary_label_pairs(pairs_df: pd.DataFrame, selected_categories: list[str]) -> pd.DataFrame:
    selected_set = set(selected_categories)

    business_raw = (
        pairs_df[["business_id", "raw_categories"]]
        .drop_duplicates("business_id")
        .copy()
    )

    business_raw["primary_subcategory"] = business_raw["raw_categories"].apply(
        lambda raw: primary_subcategory_from_raw_categories(raw, selected_categories)
    )

    primary_map = dict(
        zip(
            business_raw["business_id"],
            business_raw["primary_subcategory"]
        )
    )

    primary_pairs = pairs_df.copy()
    primary_pairs["primary_subcategory"] = primary_pairs["business_id"].map(primary_map)

    primary_pairs = primary_pairs[
        primary_pairs["subcategory"] == primary_pairs["primary_subcategory"]
    ].copy()

    primary_pairs = primary_pairs[
        primary_pairs["subcategory"].isin(selected_set)
    ].copy()

    return primary_pairs

primary_pairs = build_primary_label_pairs(pairs_top, subcategories)

primary_counts = primary_pairs["subcategory"].value_counts()
primary_labels = [
    label
    for label in subcategories
    if primary_counts.get(label, 0) >= PRIMARY_MIN_REVIEWS
]

primary_pairs_kept = primary_pairs[
    primary_pairs["subcategory"].isin(primary_labels)
].copy()

primary_corpus_df = build_corpus_from_pairs(
    primary_pairs_kept,
    primary_labels,
    MAX_TEXT_CHARS_PER_SUBCATEGORY
)

print("Original multi-label pairs:", len(pairs_top))
print("Primary-label pairs:", len(primary_pairs))
print("Primary-label categories retained:", len(primary_labels))
display(primary_corpus_df[["subcategory", "n_reviews", "n_businesses", "avg_review_stars"]])

In [ ]:
# Compare multi-label filtered TF-IDF with primary-label filtered TF-IDF

vec_primary_tfidf, X_primary_tfidf = vectorize_corpus(
    primary_corpus_df,
    use_idf=True,
    extra_stop_words=entity_stopwords
)

S_primary_tfidf = cosine_similarity(X_primary_tfidf)

primary_matrix_path = DISHRAG_DIR / "primary_label_tfidf_entity_filtered_matrix.csv"
pd.DataFrame(
    S_primary_tfidf,
    index=primary_labels,
    columns=primary_labels
).to_csv(primary_matrix_path)

def subset_similarity_matrix(S, source_labels, target_labels):
    label_to_idx = {label: i for i, label in enumerate(source_labels)}
    indices = [label_to_idx[label] for label in target_labels]
    return S[np.ix_(indices, indices)]

common_labels = [
    label
    for label in subcategories
    if label in primary_labels
]

S_multilabel_common = subset_similarity_matrix(
    S_tfidf_filtered,
    subcategories,
    common_labels
)

S_primary_common = subset_similarity_matrix(
    S_primary_tfidf,
    primary_labels,
    common_labels
)

def topk_set(S, labels, source_idx, k=5):
    return {
        labels[j]
        for j in np.argsort(-S[source_idx])
        if j != source_idx
    }.__iter__()

def topk_list(S, labels, source_idx, k=5):
    return [
        labels[j]
        for j in np.argsort(-S[source_idx])
        if j != source_idx
    ][:k]

sensitivity_rows = []

for i, label in enumerate(common_labels):
    multi_neighbors = topk_list(S_multilabel_common, common_labels, i, TOP_K_NEIGHBORS)
    primary_neighbors = topk_list(S_primary_common, common_labels, i, TOP_K_NEIGHBORS)

    multi_set = set(multi_neighbors)
    primary_set = set(primary_neighbors)

    jaccard = (
        len(multi_set & primary_set) / len(multi_set | primary_set)
        if len(multi_set | primary_set) > 0
        else np.nan
    )

    sensitivity_rows.append({
        "subcategory": label,
        "multilabel_top_neighbors": ", ".join(multi_neighbors),
        "primary_label_top_neighbors": ", ".join(primary_neighbors),
        "topk_jaccard": jaccard,
        "same_top1": multi_neighbors[0] == primary_neighbors[0] if multi_neighbors and primary_neighbors else False,
    })

primary_sensitivity_df = pd.DataFrame(sensitivity_rows)

primary_sensitivity_path = DISHRAG_DIR / "multilabel_vs_primary_label_sensitivity.csv"
primary_sensitivity_df.to_csv(primary_sensitivity_path, index=False)

print("Saved:", primary_sensitivity_path)
display(primary_sensitivity_df)

In [ ]:
# Pair-level sensitivity for important cuisine relationships

KEY_PAIRS = [
    ("Pizza", "Italian"),
    ("Sushi Bars", "Japanese"),
    ("Mediterranean", "Greek"),
    ("Mexican", "Tex-Mex"),
    ("Chinese", "Asian Fusion"),
    ("Mediterranean", "Middle Eastern"),
]

pair_sensitivity_rows = []

common_label_to_idx = {
    label: i
    for i, label in enumerate(common_labels)
}

for a, b in KEY_PAIRS:
    if a in common_label_to_idx and b in common_label_to_idx:
        i = common_label_to_idx[a]
        j = common_label_to_idx[b]

        multi_score = float(S_multilabel_common[i, j])
        primary_score = float(S_primary_common[i, j])

        pair_sensitivity_rows.append({
            "pair": f"{a} ↔ {b}",
            "multilabel_filtered_tfidf": multi_score,
            "primary_label_filtered_tfidf": primary_score,
            "score_delta_multilabel_minus_primary": multi_score - primary_score,
        })

pair_sensitivity_df = pd.DataFrame(pair_sensitivity_rows)

pair_sensitivity_path = DISHRAG_DIR / "key_pair_multilabel_sensitivity.csv"
pair_sensitivity_df.to_csv(pair_sensitivity_path, index=False)

print("Saved:", pair_sensitivity_path)
display(pair_sensitivity_df)

In [ ]:
# Bootstrap stability for DishRAG category neighbors

def bootstrap_pairs_by_subcategory(
    pairs_df: pd.DataFrame,
    labels: list[str],
    random_seed: int
) -> pd.DataFrame:
    rng = np.random.default_rng(random_seed)
    boot_parts = []

    for label in labels:
        group = pairs_df[pairs_df["subcategory"] == label].copy()
        group = group.drop_duplicates("review_id")

        if group.empty:
            continue

        sampled_indices = rng.choice(
            group.index.to_numpy(),
            size=len(group),
            replace=True
        )

        boot_parts.append(group.loc[sampled_indices].copy())

    return pd.concat(boot_parts, ignore_index=True)

def topk_neighbors_from_matrix(S, labels, k):
    output = {}

    for i, label in enumerate(labels):
        output[label] = [
            labels[j]
            for j in np.argsort(-S[i])
            if j != i
        ][:k]

    return output

original_topk = topk_neighbors_from_matrix(
    S_hybrid,
    subcategories,
    BOOTSTRAP_TOP_K
)

bootstrap_counts = {
    source: Counter()
    for source in subcategories
}

for b in range(N_BOOTSTRAPS):
    boot_pairs = bootstrap_pairs_by_subcategory(
        pairs_top,
        subcategories,
        RANDOM_SEED + b + 1000
    )

    boot_corpus_df = build_corpus_from_pairs(
        boot_pairs,
        subcategories,
        MAX_TEXT_CHARS_PER_SUBCATEGORY
    )

    vec_boot, X_boot = vectorize_corpus(
        boot_corpus_df,
        use_idf=True,
        extra_stop_words=entity_stopwords
    )

    _, S_boot_hybrid = build_hybrid_cosine_matrix(
        X_emb,
        X_boot,
        HYBRID_EMBEDDING_WEIGHT,
        HYBRID_TFIDF_WEIGHT,
    )

    boot_topk = topk_neighbors_from_matrix(
        S_boot_hybrid,
        subcategories,
        BOOTSTRAP_TOP_K
    )

    for source, neighbors in boot_topk.items():
        bootstrap_counts[source].update(neighbors)

stability_rows = []

for source in subcategories:
    original_neighbors = original_topk[source]

    for rank, neighbor in enumerate(original_neighbors, start=1):
        freq = bootstrap_counts[source][neighbor] / N_BOOTSTRAPS

        stability_rows.append({
            "source_category": source,
            "neighbor_category": neighbor,
            "original_hybrid_rank": rank,
            "bootstrap_topk_frequency": freq,
            "is_stable": freq >= STABILITY_THRESHOLD,
        })

bootstrap_stability_df = pd.DataFrame(stability_rows)

bootstrap_stability_path = DISHRAG_DIR / "hybrid_neighbor_bootstrap_stability.csv"
bootstrap_stability_df.to_csv(bootstrap_stability_path, index=False)

print("Saved:", bootstrap_stability_path)
display(bootstrap_stability_df.head(50))

In [ ]:
# Optional: HDBSCAN parameter sensitivity for robustness appendix

def run_hdbscan_with_params(X, min_cluster_size, min_samples):
    try:
        import hdbscan as _hdbscan

        model = _hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric="euclidean"
        )

        labels = model.fit_predict(X)

        return {
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
            "n_clusters_excluding_noise": cluster_count(labels),
            "noise_points": noise_count(labels),
            "silhouette": cluster_quality(X, labels, metric="cosine"),
            "labels": labels,
        }

    except ImportError:
        return {
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
            "n_clusters_excluding_noise": np.nan,
            "noise_points": np.nan,
            "silhouette": np.nan,
            "labels": None,
        }

hdbscan_sensitivity_rows = []

for min_cluster_size in [2, 3, 4, 5]:
    for min_samples in [1, 2, 3]:
        result = run_hdbscan_with_params(
            X_emb,
            min_cluster_size=min_cluster_size,
            min_samples=min_samples
        )

        hdbscan_sensitivity_rows.append({
            "min_cluster_size": result["min_cluster_size"],
            "min_samples": result["min_samples"],
            "n_clusters_excluding_noise": result["n_clusters_excluding_noise"],
            "noise_points": result["noise_points"],
            "silhouette": result["silhouette"],
        })

hdbscan_sensitivity_df = pd.DataFrame(hdbscan_sensitivity_rows)

hdbscan_sensitivity_path = DISHRAG_DIR / "hdbscan_parameter_sensitivity.csv"
hdbscan_sensitivity_df.to_csv(hdbscan_sensitivity_path, index=False)

print("Saved:", hdbscan_sensitivity_path)
display(hdbscan_sensitivity_df)

In [ ]:
# Xiaofei
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from IPython.display import Image as IPythonImage, display

OUTPUT_DIR = Path('../artifacts/task2/dishrag_category_retrieval') if Path.cwd().name == 'notebooks' else Path('artifacts/task2/dishrag_category_retrieval')
BASE_DIR = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1) Alternative similarity function: binary term-set Jaccard over aggregated subcategory corpora.
corpus = pd.read_parquet(BASE_DIR / 'artifacts/task2/task2_subcategory_corpus.parquet')
labels = corpus['subcategory'].tolist()
vectorizer = CountVectorizer(stop_words='english', ngram_range=(1, 2), max_features=6000, max_df=0.95, binary=True)
X_binary = vectorizer.fit_transform(corpus['text'].fillna(''))
intersection = (X_binary @ X_binary.T).astype(float).toarray()
term_counts = np.asarray(X_binary.sum(axis=1)).ravel()
union = term_counts[:, None] + term_counts[None, :] - intersection
S_jaccard = np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)
np.fill_diagonal(S_jaccard, 1.0)
pd.DataFrame(S_jaccard, index=labels, columns=labels).to_csv(OUTPUT_DIR / 'subcategory_similarity_binary_jaccard.csv')

pairs = []
for i, source in enumerate(labels):
    for j, neighbor in enumerate(labels):
        if j > i:
            pairs.append({'pair': f'{source} ↔ {neighbor}', 'binary_jaccard': float(S_jaccard[i, j])})
jaccard_top_pairs = pd.DataFrame(pairs).sort_values('binary_jaccard', ascending=False).head(10)
jaccard_top_pairs.to_csv(OUTPUT_DIR / 'binary_jaccard_top_pairs.csv', index=False)

# 2) Multiple clustering algorithms and very different cluster counts: KMeans vs Agglomerative, k=3 vs k=8.
S_emb = pd.read_csv(BASE_DIR / 'artifacts/task2/subcategory_similarity_embedding.csv', index_col=0)
labels = S_emb.index.tolist()
S = np.clip((S_emb.to_numpy(dtype=float) + S_emb.to_numpy(dtype=float).T) / 2, -1, 1)
D = np.clip(1 - S, 0, None)
np.fill_diagonal(D, 0.0)
profile_features = S.copy()
layout_2d = PCA(n_components=2, random_state=3847).fit_transform(profile_features)

rows = []
label_columns = {'subcategory': labels}
for algorithm in ['KMeans', 'Agglomerative']:
    for k in [3, 8]:
        if algorithm == 'KMeans':
            cluster_labels = KMeans(n_clusters=k, random_state=3847, n_init=30).fit_predict(profile_features)
        else:
            try:
                model = AgglomerativeClustering(n_clusters=k, metric='precomputed', linkage='average')
            except TypeError:
                model = AgglomerativeClustering(n_clusters=k, affinity='precomputed', linkage='average')
            cluster_labels = model.fit_predict(D)
        silhouette = silhouette_score(D, cluster_labels, metric='precomputed')
        sizes = pd.Series(cluster_labels).value_counts().sort_index().to_dict()
        rows.append({
            'algorithm': algorithm,
            'k': k,
            'silhouette_precomputed_distance': float(silhouette),
            'cluster_sizes': '; '.join(f'{int(cluster)}:{int(size)}' for cluster, size in sizes.items()),
        })
        label_columns[f'{algorithm.lower()}_k{k}'] = cluster_labels

comparison = pd.DataFrame(rows)
comparison.to_csv(OUTPUT_DIR / 'cluster_algorithm_k_comparison.csv', index=False)
labels_df = pd.DataFrame(label_columns)
labels_df.to_csv(OUTPUT_DIR / 'cluster_algorithm_k_labels.csv', index=False)
ari = adjusted_rand_score(labels_df['kmeans_k3'], labels_df['agglomerative_k3'])
(OUTPUT_DIR / 'cluster_algorithm_k_notes.txt').write_text(
    f'KMeans k=3 vs Agglomerative k=3 adjusted Rand index: {ari:.3f}\n'
    'KMeans uses each category row of the embedding-cosine matrix as a similarity-profile feature vector; '
    'Agglomerative uses precomputed distance = 1 - cosine.\n',
    encoding='utf-8',
)

# 3) Save a compact comparison figure without relying on matplotlib.
palette = ['#4E79A7', '#F28E2B', '#59A14F', '#E15759', '#B07AA1', '#76B7B2', '#EDC948', '#FF9DA7', '#9C755F', '#BAB0AC']
canvas_width, canvas_height = 1800, 1350
image = Image.new('RGB', (canvas_width, canvas_height), 'white')
draw = ImageDraw.Draw(image)
try:
    title_font = ImageFont.truetype('/System/Library/Fonts/Supplemental/Arial.ttf', 34)
    panel_font = ImageFont.truetype('/System/Library/Fonts/Supplemental/Arial.ttf', 24)
    label_font = ImageFont.truetype('/System/Library/Fonts/Supplemental/Arial.ttf', 14)
except Exception:
    title_font = panel_font = label_font = ImageFont.load_default()

draw.text((canvas_width // 2, 35), 'Clustering validation: multiple algorithms and very different k values', anchor='mm', fill='#222222', font=title_font)
panels = [
    ('KMeans', 3, 'kmeans_k3', (70, 100, 870, 650)),
    ('KMeans', 8, 'kmeans_k8', (930, 100, 1730, 650)),
    ('Agglomerative', 3, 'agglomerative_k3', (70, 730, 870, 1280)),
    ('Agglomerative', 8, 'agglomerative_k8', (930, 730, 1730, 1280)),
]
x_values, y_values = layout_2d[:, 0], layout_2d[:, 1]
xmin, xmax = x_values.min(), x_values.max()
ymin, ymax = y_values.min(), y_values.max()
for algorithm, k, column, box in panels:
    x0, y0, x1, y1 = box
    draw.rounded_rectangle(box, radius=16, outline='#CCCCCC', width=2, fill='#FAFAFA')
    silhouette = comparison[(comparison['algorithm'] == algorithm) & (comparison['k'] == k)]['silhouette_precomputed_distance'].iloc[0]
    draw.text(((x0 + x1) // 2, y0 + 28), f'{algorithm}, k={k}, silhouette={silhouette:.3f}', anchor='mm', fill='#222222', font=panel_font)
    pad = 65
    px = x0 + pad + (x_values - xmin) * ((x1 - x0 - 2 * pad) / max(xmax - xmin, 1e-9))
    py = y1 - pad - (y_values - ymin) * ((y1 - y0 - 2 * pad) / max(ymax - ymin, 1e-9))
    clusters = labels_df[column].to_numpy()
    for idx, label in enumerate(labels):
        color = palette[int(clusters[idx]) % len(palette)]
        draw.ellipse((px[idx] - 9, py[idx] - 9, px[idx] + 9, py[idx] + 9), fill=color, outline='white', width=2)
        draw.text((px[idx] + 11, py[idx] - 8), label, fill='#333333', font=label_font)
figure_path = OUTPUT_DIR / 'cluster_algorithm_k_comparison.png'
image.save(figure_path)

print('Alternative similarity top pairs:')
print(jaccard_top_pairs.head(5).to_string(index=False))
print('\nMultiple clustering algorithm / k comparison:')
print(comparison.to_string(index=False))
print('\nSaved figure:', figure_path)
display(IPythonImage(filename=str(figure_path)))


In [ ]:
# Clean noisy/brand-heavy TF-IDF top terms in main Task 2 handoff artifacts
from pathlib import Path
from collections import Counter
import re
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

BASE_DIR = Path("..") if Path.cwd().name == "notebooks" else Path(".")
TASK2_OUTPUT_DIR = BASE_DIR / "artifacts" / "task2"
DISHRAG_DIR = TASK2_OUTPUT_DIR / "dishrag_category_retrieval"
DISHRAG_DIR.mkdir(parents=True, exist_ok=True)

corpus_for_terms = pd.read_parquet(TASK2_OUTPUT_DIR / "task2_subcategory_corpus.parquet")
pairs_for_entities = pd.read_parquet(TASK2_OUTPUT_DIR / "task2_review_subcategory_pairs.parquet")
labels_for_terms = corpus_for_terms["subcategory"].tolist()

GENERIC_BUSINESS_WORDS_CLEAN = {
    "restaurant", "restaurants", "food", "foods", "bar", "bars", "cafe", "coffee", "tea",
    "grill", "grille", "kitchen", "bistro", "house", "place", "places", "shop", "market", "diner",
    "the", "and", "inc", "llc", "co", "company", "express", "factory", "buffet", "club",
    "city", "fine", "gourmet", "authentic", "fresh", "tasty", "famous", "good", "great", "best",
    "cuisine", "pizzeria", "ristorante", "lounge", "deli", "palace", "garden", "sports", "original",
    "little", "big", "new", "old", "red", "blue", "green", "golden", "silver", "king", "queen",
    "don", "los", "las", "san", "el", "la", "le", "de", "di", "da", "del", "dos", "casa",
    "cantina", "wok", "wine", "room", "joint", "joints", "spot", "spots",
    # Review/action words that otherwise create vague n-grams such as "favorite pizza" or "sushi just".
    "quality", "better", "like", "likes", "make", "made", "minute", "minutes", "pound", "pounds", "france",
    "order", "ordered", "ordering", "orders", "favorite", "favorites", "love", "loved", "just",
    "really", "pretty", "super", "nice", "amazing", "awesome", "always", "never", "eat", "eating", "ate",
    "try", "tried", "got", "came", "come", "went", "going", "ve", "ll", "didn", "doesn", "isn", "wasn",
    "yelp", "review", "reviews", "craving", "satisfy", "satisfies", "service", "served", "serving",
    "staff", "friendly", "typical", "average", "excellent", "sat", "hour", "dinner", "lunch", "want", "know",
    "speak", "spanish", "african",
}

PROTECTED_CUISINE_AND_DISH_TERMS = {
    "pizza", "mexican", "sushi", "chinese", "italian", "japanese", "thai", "greek",
    "mediterranean", "indian", "korean", "vietnamese", "barbeque", "bbq", "seafood",
    "burger", "burgers", "french", "tex", "mex", "latin", "american", "middle", "eastern",
    "vegetarian", "asian", "fusion", "cajun", "creole", "caribbean", "southern", "steakhouse",
    "steakhouses", "asada", "carne", "taco", "tacos", "burrito", "burritos", "salsa",
    "guacamole", "guac", "pico", "fajita", "fajitas", "enchilada", "enchiladas", "brisket", "ribs", "rib",
    "pulled", "pork", "gyro", "gyros", "tzatziki", "falafel", "hummus", "shawarma", "naan",
    "masala", "curry", "paneer", "tandoori", "pho", "banh", "ramen", "nigiri", "miso",
    "basil", "noodle", "noodles", "dumpling", "dumplings", "pasta", "alfredo", "risotto",
    "crawfish", "oyster", "shrimp", "clam", "chowder", "plantain", "plantains", "jerk",
    "gumbo", "jambalaya", "caviar", "souffle", "foie", "frites", "tapas", "dim", "sum",
    "mongolian", "beef", "bibimbap", "bulgogi", "kimchee", "kimchi", "banchan", "galbi",
    "tofu", "pierogi", "pupusa", "pupusas", "colombian", "salvadorean", "salvadoran", "salvadorian",
    "pernil", "pepperoni", "prosciutto", "gnocchi", "spaghetti", "meatballs", "caprese", "horchata",
    "chimichanga", "refried", "beans", "tempura", "yellowtail", "uni", "eel", "wagyu", "porterhouse",
    "oysters", "lobster", "crab", "souvlaki", "lebanese", "lentil", "kafta", "saag", "korma",
    "tikka", "vindaloo", "lassi", "congee", "duck", "tuna", "potato", "bisque", "coconut",
    "mardi", "gras", "etouffee", "etoufee", "alligator", "jamaican", "cuban", "hush", "puppies",
    "collard", "greens", "grits", "waffles", "sour", "kung", "pao", "tso", "general", "tail",
    "albacore", "calzone", "calzones", "bernaise", "kebob", "yunnan", "wontons", "dolsot", "cornbread",
}

MANUAL_NOISE_AND_BRAND_STOPWORDS = {
    "benihana", "whataburger", "fatburger", "outback", "changs", "pf", "pappadeaux", "taiko",
    "cherryblossom", "payard", "savoy", "ticoz", "bomberos", "rumbi", "emeril", "commander",
    "lucille", "yupha", "chevy", "dave", "mortons", "morton", "wtf", "doot", "robin", "lenny",
    "pei", "wei", "flo", "dan", "tb", "richardson", "tejas", "abuelo", "andre", "razz",
    "pamplemousse", "atelier", "greene", "tina", "malee", "zoe", "noto", "liliana", "louisianne",
    "jolly", "bob", "eddie", "bahama", "roy", "wasabi", "wendy", "wendys", "jungle",
    "sweet", "tomatoes", "mr", "kool", "aid", "mrs", "white", "lo", "nobu", "dragon",
    "bamboo", "cocina", "baja", "china", "chicago", "box", "brothers", "serrano", "rosati",
    "joe", "yup", "haha", "lol", "capital", "fleming", "roberto", "bianco", "ruth", "chris",
    "kona", "carl", "aquaknox", "michael", "mina", "archi", "trader", "vic", "chino", "eldorado",
    "haji", "zoes", "lolo", "biz_photos", "select", "nurses", "russell", "tiki", "japonais",
    "bobby", "flay", "ra", "jr", "fogo", "mastro", "mastros", "stingray", "flicka", "genghis",
    "breadfruit", "havana", "shabu", "mandalay", "kings", "ninja", "samurai", "sapporo", "tokyo",
    "osaka", "fuji", "sakura", "sake", "thaihouse", "angels", "bouchon", "mon", "ami", "gabi",
    "paris", "napoleon", "beijing", "shanghai", "hong", "kong", "mandarin", "panda", "pita",
    "paradise", "habana", "metro", "bobo", "brazil", "baby", "kay", "khai", "hoan", "coup", "des",
    "rally", "est", "ali", "ocha", "hap", "henry", "chambord", "chodang", "whites", "papusa",
    "kaizen", "caballero", "glendale", "cave", "creek", "redendo", "macayo", "filiberto", "garduno",
    "paymon", "humberto", "garcia", "garcias", "rubio", "keg", "casbah", "bear", "popeyes",
    "boiling", "pound", "pounds", "domino", "dominos", "burt", "foo", "rad", "dakao", "maggie", "mao", "thi", "ycs",
}

BLOCKED_EXACT_TERMS = {"thai pad"}

PREFER_PHRASE_TOKENS = {
    "kung", "pao", "tso", "general", "tex", "mardi", "hush", "collard", "foie", "tikka", "masala",
    "refried", "beans", "carne", "asada", "sour", "dim", "sum", "miso", "garlic", "thai", "basil",
    "pad", "mexican", "middle", "eastern", "korean", "cajun", "jerk", "coconut", "shrimp", "clam",
    "chowder", "mongolian", "beef", "pulled", "pork", "rib", "ribs", "gyro", "grape", "leaves",
}

def _tokenize_business_name(name: str) -> list[str]:
    return [tok.strip("'") for tok in re.findall(r"[a-z][a-z']+", str(name).lower()) if len(tok.strip("'")) >= 3]

name_counter = Counter()
if "business_name" in pairs_for_entities.columns:
    for name in pairs_for_entities["business_name"].dropna().astype(str).unique():
        for token in _tokenize_business_name(name):
            if token not in GENERIC_BUSINESS_WORDS_CLEAN:
                name_counter[token] += 1

frequent_business_name_tokens = {
    token
    for token, count in name_counter.items()
    if count >= 2 and token not in PROTECTED_CUISINE_AND_DISH_TERMS
}

clean_stopwords = (
    set(ENGLISH_STOP_WORDS)
    | GENERIC_BUSINESS_WORDS_CLEAN
    | MANUAL_NOISE_AND_BRAND_STOPWORDS
    | frequent_business_name_tokens
)
clean_stopwords = {tok for tok in clean_stopwords if tok not in PROTECTED_CUISINE_AND_DISH_TERMS}

clean_vectorizer = TfidfVectorizer(
    max_features=6000,
    min_df=1,
    max_df=0.95,
    ngram_range=(1, 2),
    stop_words=sorted(clean_stopwords),
    use_idf=True,
    sublinear_tf=True,
    norm="l2",
    token_pattern=r"(?u)\b[a-z][a-z][a-z]+\b",
)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="Your stop_words may be inconsistent")
    X_clean_terms = clean_vectorizer.fit_transform(corpus_for_terms["text"].fillna(""))
feature_terms = np.array(clean_vectorizer.get_feature_names_out())

def clean_top_terms_for_row(row, n=8):
    values = row.toarray().ravel()
    ranked = values.argsort()[::-1]
    selected = []
    selected_phrase_tokens = set()
    selected_phrase_token_sets = set()
    phrase_candidates_by_token = {}

    for idx in ranked:
        if values[idx] <= 0:
            break
        term = str(feature_terms[idx])
        tokens = term.split()
        if len(tokens) <= 1 or any(tok in clean_stopwords for tok in tokens) or len(set(tokens)) < len(tokens):
            continue
        for token in tokens:
            phrase_candidates_by_token.setdefault(token, []).append(term)

    for idx in ranked:
        if values[idx] <= 0:
            break
        term = str(feature_terms[idx])
        tokens = term.split()
        if term in BLOCKED_EXACT_TERMS:
            continue
        if any(tok in clean_stopwords for tok in tokens):
            continue
        if len(set(tokens)) < len(tokens):
            continue
        if len(tokens) == 1:
            if term in selected_phrase_tokens:
                continue
            if term in PREFER_PHRASE_TOKENS and phrase_candidates_by_token.get(term):
                continue
        else:
            token_set = frozenset(tokens)
            if token_set in selected_phrase_token_sets:
                continue
        selected.append(term)
        if len(tokens) > 1:
            selected_phrase_tokens.update(tokens)
            selected_phrase_token_sets.add(frozenset(tokens))
        if len(selected) >= n:
            break
    return "; ".join(selected)

clean_top_terms_df = pd.DataFrame({
    "subcategory": labels_for_terms,
    "top_terms": [clean_top_terms_for_row(X_clean_terms.getrow(i), n=8) for i in range(X_clean_terms.shape[0])],
})

root_clean_terms_path = TASK2_OUTPUT_DIR / "task2_top_tfidf_terms_by_subcategory.csv"
unfiltered_terms_path = TASK2_OUTPUT_DIR / "task2_top_tfidf_terms_by_subcategory_unfiltered.csv"
if root_clean_terms_path.exists() and not unfiltered_terms_path.exists():
    pd.read_csv(root_clean_terms_path).to_csv(unfiltered_terms_path, index=False)
clean_top_terms_df.to_csv(root_clean_terms_path, index=False)
clean_top_terms_df.to_csv(TASK2_OUTPUT_DIR / "task2_top_tfidf_terms_entity_filtered_by_subcategory.csv", index=False)
clean_top_terms_df.to_csv(DISHRAG_DIR / "top_tfidf_terms_entity_filtered_by_subcategory.csv", index=False)

obvious_noise_tokens = MANUAL_NOISE_AND_BRAND_STOPWORDS | {
    "benihana", "whataburger", "outback", "fatburger", "wtf", "cherryblossom", "pappadeaux",
    "fogo", "mastros", "mastro", "stingray", "flicka", "genghis", "rubio", "popeyes",
}

def _noise_hits(series):
    hits = []
    for subcat, text in series.items():
        tokens = set(re.findall(r"[a-z]+", str(text).lower()))
        overlap = sorted(tokens & obvious_noise_tokens)
        if overlap:
            hits.append((subcat, overlap))
    return hits

updates = []
for artifact_name in ["cuisine_map_summary.csv", "subcategory_clusters.csv"]:
    path = TASK2_OUTPUT_DIR / artifact_name
    df = pd.read_csv(path)
    # Keep required handoff CSVs clean: the raw unfiltered terms are preserved separately
    # in task2_top_tfidf_terms_by_subcategory_unfiltered.csv.
    df = df.drop(columns=["top_terms", "top_terms_unfiltered"], errors="ignore")
    df = df.merge(clean_top_terms_df, on="subcategory", how="left")
    df["top_terms_filtering"] = "entity_filtered_tfidf"

    if artifact_name == "cuisine_map_summary.csv":
        preferred = [
            "subcategory", "n_reviews", "n_businesses", "avg_review_stars",
            "top_terms", "top_terms_filtering", "hdbscan_cluster",
        ]
    else:
        preferred = [
            "subcategory", "n_reviews", "n_businesses", "hdbscan_cluster",
            "top_terms", "top_terms_filtering",
        ]
    df = df[[c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]]
    df.to_csv(path, index=False)
    updates.append((artifact_name, _noise_hits(df.set_index("subcategory")["top_terms"])))

top_terms_map = dict(zip(clean_top_terms_df["subcategory"], clean_top_terms_df["top_terms"]))
support_updates = []

cluster_summary_path = TASK2_OUTPUT_DIR / "cluster_summary_hdbscan.csv"
if cluster_summary_path.exists():
    cluster_summary_df = pd.read_csv(cluster_summary_path)
    if {"subcategories", "representative_terms"}.issubset(cluster_summary_df.columns):
        def _split_subcategory_list(value):
            return [item.strip() for item in str(value).split(",") if item.strip()]

        def _representative_terms_for_cluster(value, max_categories=4):
            terms = [top_terms_map.get(subcat, "") for subcat in _split_subcategory_list(value)[:max_categories]]
            return " | ".join(term for term in terms if term)

        cluster_summary_df["representative_terms"] = cluster_summary_df["subcategories"].map(_representative_terms_for_cluster)
        cluster_summary_df["representative_terms_filtering"] = "entity_filtered_tfidf"
        cluster_summary_df.to_csv(cluster_summary_path, index=False)
        support_updates.append(cluster_summary_path.name)

hybrid_neighbors_path = DISHRAG_DIR / "hybrid_category_neighbors_topk.csv"
if hybrid_neighbors_path.exists():
    hybrid_neighbors_df = pd.read_csv(hybrid_neighbors_path)
    if {"source_category", "neighbor_category"}.issubset(hybrid_neighbors_df.columns):
        hybrid_neighbors_df["source_filtered_terms"] = hybrid_neighbors_df["source_category"].map(top_terms_map).fillna("")
        hybrid_neighbors_df["neighbor_filtered_terms"] = hybrid_neighbors_df["neighbor_category"].map(top_terms_map).fillna("")
        hybrid_neighbors_df["filtered_terms_source"] = "entity_filtered_tfidf"
        hybrid_neighbors_df.to_csv(hybrid_neighbors_path, index=False)
        support_updates.append(str(hybrid_neighbors_path.relative_to(TASK2_OUTPUT_DIR)))

print("Cleaned main Task 2 top terms")
print("Entity/name stopwords used:", len(clean_stopwords))
print("Saved filtered top terms:", root_clean_terms_path)
print("Synchronized support artifacts:", support_updates if support_updates else "none found")
for artifact_name, hits in updates:
    print(f"{artifact_name}: obvious-noise hits in filtered top_terms = {len(hits)}")
    if hits:
        print("  sample hits:", hits[:5])
print("Sample cleaned top terms:")
print(clean_top_terms_df.head(25).to_string(index=False, max_colwidth=120))


In [ ]:
# Final Task 2 validation (post-extension)
from pathlib import Path
import re
import numpy as np
import pandas as pd

BASE_DIR = Path("..") if Path.cwd().name == "notebooks" else Path(".")
TASK2_OUTPUT_DIR = BASE_DIR / "artifacts" / "task2"

required_artifacts = [
    "subcategory_similarity_tfidf.csv",
    "subcategory_similarity_embedding.csv",
    "subcategory_clusters.csv",
    "cuisine_map_summary.csv",
]
optional_artifacts = [
    "task2_sampling_diagnostics.csv",
    "task2_sampling_top_businesses.csv",
    "task2_top_tfidf_terms_by_subcategory.csv",
    "task2_top_tfidf_terms_by_subcategory_unfiltered.csv",
    "task2_top_tfidf_terms_entity_filtered_by_subcategory.csv",
    "task2_comparison_summary.csv",
    "similarity_method_comparison.csv",
    "task2_reviews_subset.parquet",
    "task2_review_subcategory_pairs.parquet",
    "task2_subcategory_corpus.parquet",
]

missing = [name for name in required_artifacts if not (TASK2_OUTPUT_DIR / name).exists()]
empty = [name for name in required_artifacts if (TASK2_OUTPUT_DIR / name).exists() and (TASK2_OUTPUT_DIR / name).stat().st_size == 0]
assert not missing, f"Missing required artifacts: {missing}"
assert not empty, f"Empty required artifacts: {empty}"

def assert_similarity_matrix(path: Path, label: str):
    df = pd.read_csv(path, index_col=0)
    assert df.shape[0] == df.shape[1] and df.shape[0] > 0, f"{label} is not a non-empty square matrix: {df.shape}"
    assert list(df.index) == list(df.columns), f"{label} row and column labels differ"
    arr = df.to_numpy(dtype=float)
    assert not np.isnan(arr).any(), f"{label} contains NaN"
    assert np.allclose(arr, arr.T, atol=1e-6), f"{label} is not symmetric"
    assert np.allclose(np.diag(arr), 1.0, atol=1e-5), f"{label} diagonal is not 1"
    offdiag = arr[~np.eye(arr.shape[0], dtype=bool)]
    return df, {
        "n": df.shape[0],
        "min_offdiag": float(offdiag.min()),
        "mean_offdiag": float(offdiag.mean()),
        "max_offdiag": float(offdiag.max()),
    }

tfidf_df, tfidf_stats = assert_similarity_matrix(TASK2_OUTPUT_DIR / "subcategory_similarity_tfidf.csv", "TF-IDF similarity")
embedding_df, embedding_stats = assert_similarity_matrix(TASK2_OUTPUT_DIR / "subcategory_similarity_embedding.csv", "Embedding similarity")
assert set(tfidf_df.index) == set(embedding_df.index), "TF-IDF and embedding matrices use different labels"

clusters_df = pd.read_csv(TASK2_OUTPUT_DIR / "subcategory_clusters.csv")
summary_df = pd.read_csv(TASK2_OUTPUT_DIR / "cuisine_map_summary.csv")
assert len(clusters_df) == len(tfidf_df), "subcategory_clusters row count does not match similarity matrices"
assert len(summary_df) == len(tfidf_df), "cuisine_map_summary row count does not match similarity matrices"
assert clusters_df["subcategory"].is_unique, "subcategory_clusters has duplicate subcategories"
assert summary_df["subcategory"].is_unique, "cuisine_map_summary has duplicate subcategories"
assert set(clusters_df["subcategory"]) == set(tfidf_df.index), "cluster labels differ from similarity labels"
assert set(summary_df["subcategory"]) == set(tfidf_df.index), "summary labels differ from similarity labels"
assert "hdbscan_cluster" in clusters_df.columns, "subcategory_clusters is missing hdbscan_cluster"

if "top_terms_filtering" in summary_df.columns:
    assert (summary_df["top_terms_filtering"] == "entity_filtered_tfidf").all(), "cuisine_map_summary top terms are not consistently filtered"
    obvious_noise = {
        "benihana", "whataburger", "outback", "fatburger", "wtf", "cherryblossom", "pappadeaux",
        "pf", "changs", "pei", "wei", "doot", "robin", "lenny", "fogo", "mastro", "mastros",
        "stingray", "flicka", "genghis", "rubio", "popeyes", "domino", "dominos", "burt", "foo", "rad", "dakao", "maggie", "mao", "thi", "ycs",
    }
    top_term_sources = {
        "cuisine_map_summary.csv": summary_df.set_index("subcategory")["top_terms"],
        "subcategory_clusters.csv": clusters_df.set_index("subcategory")["top_terms"],
    }
    root_top_terms_path = TASK2_OUTPUT_DIR / "task2_top_tfidf_terms_by_subcategory.csv"
    if root_top_terms_path.exists():
        top_term_sources[root_top_terms_path.name] = pd.read_csv(root_top_terms_path).set_index("subcategory")["top_terms"]
    cluster_summary_path = TASK2_OUTPUT_DIR / "cluster_summary_hdbscan.csv"
    if cluster_summary_path.exists():
        cluster_summary_df = pd.read_csv(cluster_summary_path)
        if "representative_terms" in cluster_summary_df.columns:
            top_term_sources[cluster_summary_path.name] = cluster_summary_df.set_index("cluster")["representative_terms"]
    hybrid_neighbors_path = TASK2_OUTPUT_DIR / "dishrag_category_retrieval" / "hybrid_category_neighbors_topk.csv"
    if hybrid_neighbors_path.exists():
        hybrid_neighbors_df = pd.read_csv(hybrid_neighbors_path)
        if "source_filtered_terms" in hybrid_neighbors_df.columns:
            top_term_sources["hybrid_source_filtered_terms"] = hybrid_neighbors_df["source_filtered_terms"]
        if "neighbor_filtered_terms" in hybrid_neighbors_df.columns:
            top_term_sources["hybrid_neighbor_filtered_terms"] = hybrid_neighbors_df["neighbor_filtered_terms"]
    noise_hits = []
    for source_name, series in top_term_sources.items():
        for subcat, text in series.items():
            tokens = set(re.findall(r"[a-z]+", str(text).lower()))
            overlap = sorted(tokens & obvious_noise)
            if overlap:
                noise_hits.append((source_name, subcat, overlap))
    assert not noise_hits, f"Filtered top_terms still contain obvious noise: {noise_hits[:5]}"

comparison_path = TASK2_OUTPUT_DIR / "task2_comparison_summary.csv"
if comparison_path.exists():
    comparison_df = pd.read_csv(comparison_path)
    assert not comparison_df.empty, "task2_comparison_summary.csv is empty"
    if "embedding_used_fallback" in comparison_df.columns:
        fallback_value = str(comparison_df.loc[0, "embedding_used_fallback"]).strip().lower()
        assert fallback_value in {"false", "0"}, "Embedding path used fallback in the final real-data run"
    if "n_subcategories" in comparison_df.columns:
        assert int(comparison_df.loc[0, "n_subcategories"]) == len(tfidf_df), "n_subcategories does not match matrices"

present_optional = [name for name in optional_artifacts if (TASK2_OUTPUT_DIR / name).exists()]

print("Final Task 2 validation")
print("Artifact directory:", TASK2_OUTPUT_DIR.resolve())
print("Required artifacts: OK")
print(f"Subcategories: {len(tfidf_df)}")
print(f"TF-IDF similarity: min={tfidf_stats['min_offdiag']:.3f}, mean={tfidf_stats['mean_offdiag']:.3f}, max={tfidf_stats['max_offdiag']:.3f}")
print(f"Embedding similarity: min={embedding_stats['min_offdiag']:.3f}, mean={embedding_stats['mean_offdiag']:.3f}, max={embedding_stats['max_offdiag']:.3f}")
print("HDBSCAN cluster counts:", clusters_df["hdbscan_cluster"].value_counts().sort_index().to_dict())
print("Top-term filtering:", summary_df.get("top_terms_filtering", pd.Series(["unfiltered"])).iloc[0])
print("Optional/support artifacts present:", len(present_optional), "of", len(optional_artifacts))
print("Final validation passed.")
